# Phase 2 — NBA Data Preparation, Cleaning & Validation

**Project:** NBA Regular-Season Game and Standings Prediction  
**Historical coverage:** 2015–16 through 2024–25  
**Environment:** Python / Google Colab / Google Drive  
**Input:** `nba_complete_dataset_2015_2025.csv` (game and player records combined)

## Research objective

Prepare quality-controlled NBA game outcomes and individual player box scores for exploratory data analysis and eventual game-win probability and standings prediction. The process diagnoses missing player observations, standardizes statistics, audits duplicate ESPN player identities, validates reported starters, reconciles player scoring to official team scores, and produces separate modeling-ready datasets.

### Workflow

1. Load and examine the combined game–player dataset.
2. Diagnose missing values and participation status.
3. Parse shooting statistics and minutes; inspect conversion failures.
4. Investigate and resolve repeated player identities.
5. Validate and correct five-player starting lineups.
6. Compare team scores against aggregated player scoring.
7. Identify placeholder box scores and independently reviewed exceptions.
8. Export the full game dataset, player dataset, performance subset, and quality/audit reports.

> **Preservation policy:** All original Python code cells and saved outputs are retained unchanged. Only explanatory Markdown has been inserted, and two existing code cells were repositioned to respect their dependencies. The documented notebook has **not** been rerun; historical outputs are snapshots from your uploaded notebook.

> **Execution note:** Two early code cells refer to `combined`, although this notebook loads the CSV into `data`. Those references can cause `NameError` on **Runtime → Run all** in a fresh session. They remain unchanged as requested. Review them in the working copy if you plan to rerun the notebook from the beginning; no fixes have been silently applied.

---

## 2.1 Environment setup

**Objective:** Make Google Drive available to Colab. The subsequent file-loading and export operations use paths under `/content/drive`.

**Method:** Mount Google Drive before loading local CSV files.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


## 2.2 Load and inspect the combined dataset

**Objective:** Establish the number of records, variables, games, and players; review data types and column-level missingness.

**Source:** `nba_complete_dataset_2015_2025.csv` from Phase 1. The CSV is loaded into **`data`**, with game and player IDs explicitly read as strings.

**Important pre-existing code reference:** The missing-value summary in this cell uses `combined` rather than `data`; the exact code has been preserved. This may fail if `combined` is not defined in the current session.

In [ ]:
import pandas as pd
from pathlib import Path

BASE_DIR = Path(
    "/content/drive/MyDrive/Colab Notebooks/Dataset/NBA Dataset"
)

data = pd.read_csv(
    BASE_DIR / "nba_complete_dataset_2015_2025.csv",
    dtype={
        "GAME_ID": str,
        "TEAM_ID": str,
        "PLAYER_ID": str
    }
)

print("DATASET SUMMARY")
print("=" * 50)
print("Rows:", data.shape[0])
print("Columns:", data.shape[1])
print("Unique games:", data["GAME_ID"].nunique())
print("Unique players:", data["PLAYER_ID"].nunique())

print("\nCOLUMN NAMES")
print(data.columns.tolist())

print("\nDATA TYPES")
print(data.dtypes)

print("\nMISSING VALUES SUMMARY")
print("=" * 60)

missing_summary = pd.DataFrame({
    "Missing Count": combined.isna().sum(),
    "Missing Percentage (%)": (
        combined.isna().mean() * 100
    ).round(2)
})

missing_summary = missing_summary.sort_values(
    by="Missing Percentage (%)",
    ascending=False
)

display(missing_summary)


print("\nSAMPLE RECORDS")
data.head()

/tmp/ipykernel_5037/790821502.py:8: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv(


DATASET SUMMARY
Rows: 308263
Columns: 32
Unique games: 11979
Unique players: 1488

COLUMN NAMES
['SEASON', 'GAME_ID', 'GAME_DATE', 'TEAM_ID', 'TEAM', 'PLAYER_ID', 'PLAYER', 'POSITION', 'STARTER', 'DID_NOT_PLAY', 'MIN', 'PTS', 'FG', '3PT', 'FT', 'REB', 'AST', 'TO', 'STL', 'BLK', 'OREB', 'DREB', 'PF', '+/-', 'HOME_TEAM_ID', 'HOME_TEAM', 'AWAY_TEAM_ID', 'AWAY_TEAM', 'HOME_SCORE', 'AWAY_SCORE', 'HOME_WIN', 'EVENT_NAME']

DATA TYPES
SEASON           object
GAME_ID          object
GAME_DATE        object
TEAM_ID          object
TEAM             object
PLAYER_ID        object
PLAYER           object
POSITION         object
STARTER            bool
DID_NOT_PLAY       bool
MIN              object
PTS             float64
FG               object
3PT              object
FT               object
REB             float64
AST             float64
TO              float64
STL             float64
BLK             float64
OREB            float64
DREB            float64
PF              float64
+/-             

,Missing Count,Missing Percentage (%)
3PT,53499,17.35
OREB,53499,17.35
AST,53499,17.35
FT,53499,17.35
REB,53499,17.35
PTS,53499,17.35
FG,53499,17.35
MIN,53499,17.35
TO,53499,17.35
STL,53499,17.35



SAMPLE RECORDS


,SEASON,GAME_ID,GAME_DATE,TEAM_ID,TEAM,PLAYER_ID,PLAYER,POSITION,STARTER,DID_NOT_PLAY,...,PF,+/-,HOME_TEAM_ID,HOME_TEAM,AWAY_TEAM_ID,AWAY_TEAM,HOME_SCORE,AWAY_SCORE,HOME_WIN,EVENT_NAME
0,2015-16,400827888,2015-10-28T00:00Z,1,Atlanta Hawks,3015,Paul Millsap,PF,True,False,...,4.0,-22.0,1,Atlanta Hawks,8,Detroit Pistons,94,106,0,Detroit Pistons at Atlanta Hawks
1,2015-16,400827888,2015-10-28T00:00Z,1,Atlanta Hawks,6637,Kent Bazemore,SF,True,False,...,3.0,-17.0,1,Atlanta Hawks,8,Detroit Pistons,94,106,0,Detroit Pistons at Atlanta Hawks
2,2015-16,400827888,2015-10-28T00:00Z,1,Atlanta Hawks,3213,Al Horford,C,True,False,...,1.0,-5.0,1,Atlanta Hawks,8,Detroit Pistons,94,106,0,Detroit Pistons at Atlanta Hawks
3,2015-16,400827888,2015-10-28T00:00Z,1,Atlanta Hawks,4015,Jeff Teague,PG,True,False,...,1.0,-23.0,1,Atlanta Hawks,8,Detroit Pistons,94,106,0,Detroit Pistons at Atlanta Hawks
4,2015-16,400827888,2015-10-28T00:00Z,1,Atlanta Hawks,2011,Kyle Korver,G,True,False,...,4.0,-9.0,1,Atlanta Hawks,8,Detroit Pistons,94,106,0,Detroit Pistons at Atlanta Hawks


## 2.3 Missing-value and DNP investigation

**Objective:** Distinguish structurally absent player box-score statistics from unexpected missing performance values.

**Methodology:** Convert `DID_NOT_PLAY` to a Boolean, identify rows where **all 14** core statistics are null, and compare the result against the DNP indicator using a cross-tabulation.

**Execution note:** The print statement referencing `combined["ALL_STATS_MISSING"]` is retained from the uploaded code. In a fresh runtime, the loaded DataFrame is named `data`.

In [ ]:

stat_columns = [
    "FG", "3PT", "REB", "FT", "MIN", "PTS",
    "TO", "AST", "+/-", "PF", "DREB",
    "OREB", "BLK", "STL"
]

data["DID_NOT_PLAY"] = (
    data["DID_NOT_PLAY"]
    .astype(str)
    .str.lower()
    .eq("true")
)

data["ALL_STATS_MISSING"] = (
    data[stat_columns].isna().all(axis=1)
)

print("Total player records:", len(data))
print("Players marked DNP:", data["DID_NOT_PLAY"].sum())
print("Records with all statistics missing:",
      combined["ALL_STATS_MISSING"].sum())

print("\nDNP vs Missing Statistics")
display(
    pd.crosstab(
        data["DID_NOT_PLAY"],
        data["ALL_STATS_MISSING"],
        margins=True
    )
)

print("\nMissing statistics among active players:")
active = data[~data["DID_NOT_PLAY"]]

display(
    pd.DataFrame({
        "Missing Count": active[stat_columns].isna().sum(),
        "Missing Percentage (%)": (
            active[stat_columns].isna().mean() * 100
        ).round(2)
    })
)

Total player records: 308263
Players marked DNP: 53499
Records with all statistics missing: 53499

DNP vs Missing Statistics


ALL_STATS_MISSING,False,True,All
DID_NOT_PLAY,,,
False,254764,0,254764
True,0,53499,53499
All,254764,53499,308263



Missing statistics among active players:


,Missing Count,Missing Percentage (%)
FG,0,0.0
3PT,0,0.0
REB,0,0.0
FT,0,0.0
MIN,0,0.0
PTS,0,0.0
TO,0,0.0
AST,0,0.0
+/-,0,0.0
PF,0,0.0


**Observed missingness result:** 308,263 original player rows included 53,499 marked DNP; the same 53,499 had all core box-score statistics missing. The 254,764 non-DNP rows had all 14 core statistics present. Missingness is therefore structurally associated with DNP in the raw player data.

### 2.3.1 Confirm partial and unexpected missingness

This second check uses a copy of `data` to distinguish records with **all** statistics missing, **any** statistics missing, and active/non-DNP records. It provides a more targeted data-quality check before converting performance variables.

In [ ]:
import pandas as pd

# Statistics to investigate
stat_columns = [
    "MIN", "PTS", "FG", "3PT", "FT",
    "REB", "AST", "TO", "STL", "BLK",
    "OREB", "DREB", "PF", "+/-"
]

# Work on a copy to preserve the original dataset
check_data = data.copy()

# Standardize DNP indicators
check_data["DNP_FLAG"] = (
    check_data["DID_NOT_PLAY"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("true")
)

# Identify records with missing statistics
check_data["ALL_STATS_MISSING"] = (
    check_data[stat_columns].isna().all(axis=1)
)

check_data["ANY_STATS_MISSING"] = (
    check_data[stat_columns].isna().any(axis=1)
)

# Compare missing statistics with ESPN's DNP indicator
print("DNP vs Missing Statistics")
display(
    pd.crosstab(
        check_data["DNP_FLAG"],
        check_data["ALL_STATS_MISSING"],
        margins=True
    )
)

# Identify players not flagged as DNP
# but with all statistics missing
unexpected_missing = check_data.loc[
    (~check_data["DNP_FLAG"]) &
    (check_data["ALL_STATS_MISSING"])
].copy()

print("\nUnexpected missing-statistics records:",
      len(unexpected_missing))

display(
    unexpected_missing[
        [
            "SEASON", "GAME_ID", "PLAYER",
            "TEAM", "STARTER", "DID_NOT_PLAY",
            "MIN", "PTS", "REB", "AST"
        ]
    ].head(20)
)

# Check partial missing statistics among non-DNP players
partial_missing = check_data.loc[
    (~check_data["DNP_FLAG"]) &
    (check_data["ANY_STATS_MISSING"]) &
    (~check_data["ALL_STATS_MISSING"])
]

print("\nPartially missing records:",
      len(partial_missing))



DNP vs Missing Statistics


ALL_STATS_MISSING,False,True,All
DNP_FLAG,,,
False,254764,0,254764
True,0,53499,53499
All,254764,53499,308263



Unexpected missing-statistics records: 0


,SEASON,GAME_ID,PLAYER,TEAM,STARTER,DID_NOT_PLAY,MIN,PTS,REB,AST



Partially missing records: 0


## 2.4 Numerical transformation of box-score statistics

**Objective:** Produce numerical player shooting counts, shooting rates, and playing time without overwriting the raw source fields.

**Methodology:** Split strings such as `8-15` into shots made (`FGM`) and attempted (`FGA`), calculate percentages, convert common counting statistics to numbers, and parse minute values into `MIN_NUM`.

**Limitation:** Text values such as `--` cannot be interpreted as playing time; they must be diagnosed, not silently changed to zero.

Clean and convert player statistics
Your ESPN data includes statistics such as FG, 3PT, and FT, which are stored as strings like "8-15". We need to separate makes and attempts into numerical variables.

In [ ]:

import pandas as pd

# Exclude players who did not participate in the game.
# Keep the original dataset unchanged.
data["DID_NOT_PLAY"] = (
    data["DID_NOT_PLAY"]
    .astype(str)
    .str.lower()
    .eq("true")
)

active_players = data.loc[
    ~data["DID_NOT_PLAY"]
].copy()


# Convert shooting statistics from "made-attempted" strings
# into separate numerical columns.
# Example: "8-15" becomes 8 field goals made and 15 attempted.
for col, prefix in [
    ("FG", "FG"),
    ("3PT", "FG3"),
    ("FT", "FT")
]:
    split_stats = active_players[col].str.split(
        "-", n=1, expand=True
    )

    active_players[f"{prefix}M"] = pd.to_numeric(
        split_stats[0], errors="coerce"
    )

    active_players[f"{prefix}A"] = pd.to_numeric(
        split_stats[1], errors="coerce"
    )

    # Shooting percentage = shots made / shots attempted.
    # Replace zero attempts with NaN to avoid division by zero.
    active_players[f"{prefix}_PCT"] = (
        active_players[f"{prefix}M"]
        / active_players[f"{prefix}A"].replace(0, float("nan"))
    )


# Convert individual player statistics to numerical values.
numeric_columns = [
    "PTS", "REB", "AST", "TO", "STL", "BLK",
    "OREB", "DREB", "PF", "+/-"
]

for col in numeric_columns:
    active_players[col] = pd.to_numeric(
        active_players[col], errors="coerce"
    )


# Convert minutes played to decimal minutes.
# Example: "25:30" becomes 25.5 minutes.
def convert_minutes(value):
    if pd.isna(value):
        return float("nan")

    value = str(value).strip()

    if ":" in value:
        minutes, seconds = value.split(":", 1)

        try:
            return float(minutes) + float(seconds) / 60
        except ValueError:
            return float("nan")

    return pd.to_numeric(value, errors="coerce")


active_players["MIN_NUM"] = active_players["MIN"].apply(
    convert_minutes
)


# Convert game dates to datetime for chronological analysis.
active_players["GAME_DATE"] = pd.to_datetime(
    active_players["GAME_DATE"],
    errors="coerce"
)


# Validate the cleaned player statistics.
print("Active player records:", len(active_players))

print("\nConverted statistics:")
display(
    active_players[
        [
            "PLAYER", "MIN", "MIN_NUM", "PTS",
            "FG", "FGM", "FGA", "FG_PCT",
            "3PT", "FG3M", "FG3A", "FG3_PCT",
            "FT", "FTM", "FTA", "FT_PCT"
        ]
    ].head(10)
)


# Check whether any values failed during numerical conversion.
check_columns = [
    "FGM", "FGA", "FG3M", "FG3A",
    "FTM", "FTA", "MIN_NUM"
]

print("\nConversion errors:")
display(
    active_players[check_columns].isna().sum()
)

Active player records: 254764

Converted statistics:


,PLAYER,MIN,MIN_NUM,PTS,FG,FGM,FGA,FG_PCT,3PT,FG3M,FG3A,FG3_PCT,FT,FTM,FTA,FT_PCT
0,Paul Millsap,36,36.0,19.0,7-15,7,15,0.466667,2-6,2,6,0.333333,3-4,3,4,0.750000
1,Kent Bazemore,21,21.0,0.0,0-3,0,3,0.000000,0-1,0,1,0.000000,0-0,0,0,NaN
2,Al Horford,30,30.0,15.0,6-11,6,11,0.545455,1-3,1,3,0.333333,2-3,2,3,0.666667
3,Jeff Teague,32,32.0,18.0,7-16,7,16,0.437500,1-3,1,3,0.333333,3-4,3,4,0.750000
4,Kyle Korver,29,29.0,7.0,3-9,3,9,0.333333,1-5,1,5,0.200000,0-0,0,0,NaN
5,Mike Scott,5,5.0,4.0,2-4,2,4,0.500000,0-2,0,2,0.000000,0-0,0,0,NaN
6,Thabo Sefolosha,19,19.0,2.0,1-3,1,3,0.333333,0-1,0,1,0.000000,0-0,0,0,NaN
7,Tiago Splitter,16,16.0,4.0,2-5,2,5,0.400000,0-0,0,0,NaN,0-0,0,0,NaN
8,Mike Muscala,8,8.0,0.0,0-1,0,1,0.000000,0-0,0,0,NaN,0-0,0,0,NaN
9,Lamar Patterson,18,18.0,5.0,1-1,1,1,1.000000,1-1,1,1,1.000000,2-2,2,2,1.000000



Conversion errors:


,0
FGM,0
FGA,0
FG3M,0
FG3A,0
FTM,0
FTA,0
MIN_NUM,10392


**Transformation outcome:** Shooting makes and attempts converted without reported errors. `MIN_NUM` had 10,392 unconvertible or unavailable values among the initial 254,764 non-DNP rows; this was treated as a separate source-quality issue, not as missing shooting statistics.

### 2.4.1 Diagnose unavailable minutes among players marked as starters

**Objective:** Check whether `MIN_NUM` conversion failures extend to starting-lineup records. A missing minute value does not, by itself, prove a player did not participate.

In [ ]:
# Standardize the starter indicator.
active_players["STARTER_FLAG"] = (
    active_players["STARTER"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("true")
)

# Identify players marked as starters with missing minutes.
missing_starters = active_players.loc[
    active_players["STARTER_FLAG"]
    & active_players["MIN_NUM"].isna()
].copy()

print("Starters with missing minutes:", len(missing_starters))
print("Affected games:", missing_starters["GAME_ID"].nunique())

print("\nOriginal minutes values:")
display(missing_starters["MIN"].value_counts(dropna=False))

print("\nAffected seasons:")
display(
    missing_starters["SEASON"]
    .value_counts()
    .sort_index()
)

print("\nSample affected starters:")
display(
    missing_starters[
        [
            "SEASON", "GAME_DATE", "GAME_ID",
            "TEAM", "PLAYER", "STARTER",
            "MIN", "PTS", "REB", "AST"
        ]
    ].head(20)
)


Starters with missing minutes: 4873
Affected games: 490

Original minutes values:


,count
MIN,
--,4873



Affected seasons:


,count
SEASON,
2015-16,1610
2016-17,1630
2017-18,1630
2018-19,3



Sample affected starters:


,SEASON,GAME_DATE,GAME_ID,TEAM,PLAYER,STARTER,MIN,PTS,REB,AST
26,2015-16,2015-10-28 00:00:00+00:00,400827889,Chicago Bulls,Pau Gasol,True,--,0.0,0.0,0.0
27,2015-16,2015-10-28 00:00:00+00:00,400827889,Chicago Bulls,Jimmy Butler III,True,--,0.0,0.0,0.0
28,2015-16,2015-10-28 00:00:00+00:00,400827889,Chicago Bulls,Nikola Mirotic,True,--,0.0,0.0,0.0
29,2015-16,2015-10-28 00:00:00+00:00,400827889,Chicago Bulls,Tony Snell,True,--,0.0,0.0,0.0
30,2015-16,2015-10-28 00:00:00+00:00,400827889,Chicago Bulls,Derrick Rose,True,--,0.0,0.0,0.0
39,2015-16,2015-10-28 00:00:00+00:00,400827889,Cleveland Cavaliers,LeBron James,True,--,0.0,0.0,0.0
40,2015-16,2015-10-28 00:00:00+00:00,400827889,Cleveland Cavaliers,Kevin Love,True,--,0.0,0.0,0.0
41,2015-16,2015-10-28 00:00:00+00:00,400827889,Cleveland Cavaliers,Timofey Mozgov,True,--,0.0,0.0,0.0
42,2015-16,2015-10-28 00:00:00+00:00,400827889,Cleveland Cavaliers,Mo Williams,True,--,0.0,0.0,0.0
43,2015-16,2015-10-28 00:00:00+00:00,400827889,Cleveland Cavaliers,JR Smith,True,--,0.0,0.0,0.0


**Diagnostic observation:** 4,873 starter-labeled rows initially had unavailable minutes, concentrated in 2015–16 through 2017–18. That concentration anticipated the season-specific placeholder-box-score issue diagnosed later.

## 2.5 Initial starting-five consistency check

**Objective:** Count players flagged as starters for each team-game. An NBA team should have **exactly five** starters at tipoff.

**Methodology:** Create a normalized `STARTER_FLAG` and examine distributions and exceptions before changing any records. This is an **initial diagnostic**, prior to correcting repeated player identities.

In [ ]:
# Count starters per team per game.
starter_validation = (
    active_players.groupby(
        ["SEASON", "GAME_ID", "TEAM"]
    )["STARTER_FLAG"]
    .sum()
    .reset_index(name="STARTER_COUNT")
)

print("Starter count distribution:")
display(
    starter_validation["STARTER_COUNT"]
    .value_counts()
    .sort_index()
    .to_frame("Team-Games")
)

print("\nTeam-games without exactly five starters:")
display(
    starter_validation.loc[
        starter_validation["STARTER_COUNT"] != 5
    ].head(20)
)


Starter count distribution:


,Team-Games
STARTER_COUNT,
5,23938
6,20



Team-games without exactly five starters:


,SEASON,GAME_ID,TEAM,STARTER_COUNT
5473,2017-18,400975023,Minnesota Timberwolves,6
7387,2018-19,401070216,Phoenix Suns,6
7415,2018-19,401070239,Philadelphia 76ers,6
7465,2018-19,401070716,Phoenix Suns,6
7489,2018-19,401070728,Phoenix Suns,6
7515,2018-19,401070741,Phoenix Suns,6
7555,2018-19,401070761,Phoenix Suns,6
7565,2018-19,401070766,Phoenix Suns,6
7640,2018-19,401070804,Phoenix Suns,6
7669,2018-19,401070818,Phoenix Suns,6


**Initial outcome:** 23,938 team-games had five starters, while 20 had six in the initial active-player check. This motivated a duplicate-identity investigation before changing starter flags.

### 2.5.1 Examine a six-starter example

The code inspects game `401070216`, which motivated additional investigation of players appearing more than once with the same name in a team-game.

In [ ]:

# Investigate one game with six reported starters.
game_id = "401070216"

# Use the original combined dataset to avoid filtering out players.
game_check = data.loc[
    data["GAME_ID"].astype(str) == game_id
].copy()

# Standardize starter and DNP indicators.
game_check["STARTER_FLAG"] = (
    game_check["STARTER"]
    .astype(str)
    .str.lower()
    .eq("true")
)

game_check["DNP_FLAG"] = (
    game_check["DID_NOT_PLAY"]
    .astype(str)
    .str.lower()
    .eq("true")
)

# Display all players marked as starters.
display(
    game_check.loc[
        game_check["STARTER_FLAG"],
        [
            "TEAM", "PLAYER", "POSITION",
            "STARTER_FLAG", "DNP_FLAG",
            "MIN", "PTS", "REB", "AST"
        ]
    ].sort_values(["TEAM", "PLAYER"])
)


,TEAM,PLAYER,POSITION,STARTER_FLAG,DNP_FLAG,MIN,PTS,REB,AST
94315,Dallas Mavericks,DeAndre Jordan,C,True,False,31,11.0,12.0,1.0
94318,Dallas Mavericks,Dennis Smith Jr.,G,True,False,28,13.0,2.0,5.0
94314,Dallas Mavericks,Dorian Finney-Smith,F,True,False,30,12.0,3.0,1.0
94317,Dallas Mavericks,Luka Doncic,G,True,False,32,10.0,8.0,4.0
94316,Dallas Mavericks,Wesley Matthews,SG,True,False,34,15.0,2.0,5.0
94330,Phoenix Suns,Deandre Ayton,C,True,False,36,18.0,10.0,6.0
94332,Phoenix Suns,Devin Booker,G,True,False,33,35.0,4.0,7.0
94328,Phoenix Suns,Isaiah Canaan,F,True,False,34,8.0,6.0,7.0
94331,Phoenix Suns,Isaiah Canaan,G,True,False,34,8.0,6.0,7.0
94327,Phoenix Suns,Ryan Anderson,PF,True,False,24,2.0,7.0,3.0


## 2.6 Duplicate identity investigation

### 2.6.1 Exact player-ID duplicates

**Objective:** Test uniqueness of `(GAME_ID, TEAM_ID, PLAYER_ID)` in the original data. Duplicate IDs and repeated names are separate problems, so both must be checked.

In [ ]:
# Identify duplicate player records within the same game and team.
duplicate_mask = data.duplicated(
    subset=["GAME_ID", "TEAM_ID", "PLAYER_ID"],
    keep=False
)

duplicate_players = data.loc[duplicate_mask].copy()

print("Total duplicate records:", len(duplicate_players))

print("\nAffected games:")
print(duplicate_players["GAME_ID"].nunique())

print("\nAffected seasons:")
display(
    duplicate_players.groupby("SEASON")
    .agg(
        DUPLICATE_RECORDS=("PLAYER_ID", "size"),
        AFFECTED_GAMES=("GAME_ID", "nunique")
    )
)

print("\nSample duplicate players:")
display(
    duplicate_players[
        [
            "SEASON", "GAME_ID", "TEAM",
            "PLAYER_ID", "PLAYER", "POSITION",
            "STARTER", "MIN", "PTS", "REB", "AST"
        ]
    ].sort_values(
        ["GAME_ID", "TEAM", "PLAYER"]
    ).head(30)
)


Total duplicate records: 0

Affected games:
0

Affected seasons:


,DUPLICATE_RECORDS,AFFECTED_GAMES
SEASON,,



Sample duplicate players:


,SEASON,GAME_ID,TEAM,PLAYER_ID,PLAYER,POSITION,STARTER,MIN,PTS,REB,AST


### 2.6.2 Compare statistics for repeated player IDs

Compare per-player statistics within exact-ID duplicate groups. The check is retained even though the uploaded notebook's output shows no exact-ID duplicate groups.

In [ ]:

# Compare statistics among records with the same
# game, team, and player identifiers.
key_columns = ["GAME_ID", "TEAM_ID", "PLAYER_ID"]

stat_columns = [
    "MIN", "PTS", "FG", "3PT", "FT",
    "REB", "AST", "TO", "STL", "BLK",
    "OREB", "DREB", "PF", "+/-"
]

# Count distinct values within each duplicate group.
duplicate_comparison = (
    duplicate_players.groupby(
        key_columns, dropna=False
    )[stat_columns]
    .nunique(dropna=False)
)

# Flag groups where at least one statistic differs.
conflicting_groups = duplicate_comparison[
    duplicate_comparison.gt(1).any(axis=1)
]

print("Duplicate groups:",
      len(duplicate_comparison))

print("Groups with conflicting statistics:",
      len(conflicting_groups))

print("Groups with identical statistics:",
      len(duplicate_comparison) - len(conflicting_groups))

display(conflicting_groups.head(20))


Duplicate groups: 0
Groups with conflicting statistics: 0
Groups with identical statistics: 0


,,,MIN,PTS,FG,3PT,FT,REB,AST,TO,STL,BLK,OREB,DREB,PF,+/-
GAME_ID,TEAM_ID,PLAYER_ID,,,,,,,,,,,,,,


### 2.6.3 Inspect a repeated-name example

Inspect Isaiah Canaan in game `401070216`, including position and ESPN player IDs, to understand why exact-ID deduplication alone cannot resolve repeated-name records.

In [ ]:

# Examine all records for Isaiah Canaan in the affected game.
canaan_records = data.loc[
    (data["GAME_ID"].astype(str) == "401070216")
    & (data["TEAM"] == "Phoenix Suns")
    & (data["PLAYER"].str.contains(
        "Isaiah Canaan", case=False, na=False
    ))
]

display(
    canaan_records[
        [
            "GAME_ID", "TEAM_ID", "PLAYER_ID",
            "PLAYER", "POSITION", "STARTER",
            "MIN", "PTS", "REB", "AST"
        ]
    ]
)


,GAME_ID,TEAM_ID,PLAYER_ID,PLAYER,POSITION,STARTER,MIN,PTS,REB,AST
94328,401070216,21,4412182,Isaiah Canaan,F,True,34,8.0,6.0,7.0
94331,401070216,21,2490589,Isaiah Canaan,G,True,34,8.0,6.0,7.0


### 2.6.4 Identify repeated names within a team-game

Group by `(GAME_ID, TEAM_ID, PLAYER)` rather than player ID. These are **potential** identity duplicates, not automatic deletion candidates: distinct individuals can share a name.

In [ ]:

# Identify repeated player names within the same game and team.
name_keys = ["GAME_ID", "TEAM_ID", "PLAYER"]

potential_duplicates = data.loc[
    data.duplicated(
        subset=name_keys,
        keep=False
    )
].copy()

print("Potential duplicate records:", len(potential_duplicates))

print(
    "Affected games:",
    potential_duplicates["GAME_ID"].nunique()
)

print("\nAffected seasons:")
display(
    potential_duplicates.groupby("SEASON")
    .agg(
        RECORDS=("PLAYER", "size"),
        GAMES=("GAME_ID", "nunique")
    )
)

print("\nSample potential duplicates:")
display(
    potential_duplicates[
        [
            "SEASON", "GAME_ID", "TEAM",
            "PLAYER_ID", "PLAYER", "POSITION",
            "STARTER", "MIN", "PTS", "REB", "AST"
        ]
    ].sort_values(
        ["GAME_ID", "TEAM", "PLAYER"]
    ).head(30)
)

Potential duplicate records: 72
Affected games: 36

Affected seasons:


,RECORDS,GAMES
SEASON,,
2018-19,64,32
2019-20,8,4



Sample potential duplicates:


,SEASON,GAME_ID,TEAM,PLAYER_ID,PLAYER,POSITION,STARTER,MIN,PTS,REB,AST
94328,2018-19,401070216,Phoenix Suns,4412182,Isaiah Canaan,F,True,34,8.0,6.0,7.0
94331,2018-19,401070216,Phoenix Suns,2490589,Isaiah Canaan,G,True,34,8.0,6.0,7.0
111576,2018-19,401070239,Philadelphia 76ers,3191,Corey Brewer,F,True,33.0,11.0,3.0,1.0
111578,2018-19,401070239,Philadelphia 76ers,4415554,Corey Brewer,F,True,33.0,11.0,3.0,1.0
94870,2018-19,401070716,Phoenix Suns,4412182,Isaiah Canaan,F,True,30,15.0,4.0,2.0
94873,2018-19,401070716,Phoenix Suns,2490589,Isaiah Canaan,G,True,30,15.0,4.0,2.0
95231,2018-19,401070728,Phoenix Suns,4412182,Isaiah Canaan,F,True,26,7.0,1.0,2.0
95234,2018-19,401070728,Phoenix Suns,2490589,Isaiah Canaan,G,True,26,7.0,1.0,2.0
95565,2018-19,401070741,Phoenix Suns,4412182,Isaiah Canaan,F,True,28,4.0,5.0,2.0
95568,2018-19,401070741,Phoenix Suns,2490589,Isaiah Canaan,G,True,28,4.0,5.0,2.0


**Repeated-name finding:** The original data contained 72 records across 36 repeated-name team-game groups, concentrated in 2018–19 (32 games) and 2019–20 (4 games). Exact-ID duplicates had not detected them.

### 2.6.5 Test whether repeated-name statistics differ

Calculate the number of distinct values for each box-score field in repeated-name groups. This initial comparison treats source strings as written; subsequent code normalizes statistical values.

In [ ]:

# Compare statistics across repeated player-name records.
stat_columns = [
    "MIN", "PTS", "FG", "3PT", "FT",
    "REB", "AST", "TO", "STL", "BLK",
    "OREB", "DREB", "PF", "+/-"
]

name_comparison = (
    potential_duplicates.groupby(
        name_keys, dropna=False
    )[stat_columns]
    .nunique(dropna=False)
)

conflicting_groups = name_comparison.loc[
    name_comparison.gt(1).any(axis=1)
]

print("Repeated-name groups:", len(name_comparison))
print("Groups with conflicting statistics:", len(conflicting_groups))
print(
    "Groups with identical statistics:",
    len(name_comparison) - len(conflicting_groups)
)

display(conflicting_groups.head(20))


Repeated-name groups: 36
Groups with conflicting statistics: 36
Groups with identical statistics: 0


,,,MIN,PTS,FG,3PT,FT,REB,AST,TO,STL,BLK,OREB,DREB,PF,+/-
GAME_ID,TEAM_ID,PLAYER,,,,,,,,,,,,,,
401070216,21,Isaiah Canaan,1,1,1,1,1,1,1,1,1,1,1,1,1,2
401070239,20,Corey Brewer,1,1,1,1,1,1,1,1,1,1,1,1,1,2
401070716,21,Isaiah Canaan,1,1,1,1,1,1,1,1,1,1,1,1,1,2
401070728,21,Isaiah Canaan,1,1,1,1,1,1,1,1,1,1,1,1,1,2
401070741,21,Isaiah Canaan,1,1,1,1,1,1,1,1,1,1,1,1,1,2
401070761,21,Isaiah Canaan,1,1,1,1,1,1,1,1,1,1,1,1,1,2
401070766,21,Isaiah Canaan,1,1,1,1,1,1,1,1,1,1,1,1,1,2
401070804,21,Isaiah Canaan,1,1,1,1,1,1,1,1,1,1,1,1,1,2
401070818,21,Isaiah Canaan,1,1,1,1,1,1,1,1,1,1,1,1,1,2


### 2.6.6 Inspect paired records side by side

Display detailed game, identity, position, lineup, and scoring information for suspected duplicates before choosing any corrections.

In [ ]:
# Identify repeated player names within the same team-game.
name_keys = ["GAME_ID", "TEAM_ID", "PLAYER"]

potential_duplicates = data.loc[
    data.duplicated(subset=name_keys, keep=False)
].copy()

# Show the full records for comparison.
comparison_columns = [
    "SEASON", "GAME_ID", "TEAM", "PLAYER",
    "PLAYER_ID", "POSITION", "STARTER",
    "DID_NOT_PLAY", "MIN", "PTS", "FG",
    "3PT", "FT", "REB", "AST", "TO",
    "STL", "BLK", "OREB", "DREB", "PF", "+/-"
]

display(
    potential_duplicates[comparison_columns]
    .sort_values(["SEASON", "GAME_ID", "TEAM", "PLAYER"])
    .head(30)
)


,SEASON,GAME_ID,TEAM,PLAYER,PLAYER_ID,POSITION,STARTER,DID_NOT_PLAY,MIN,PTS,...,FT,REB,AST,TO,STL,BLK,OREB,DREB,PF,+/-
94328,2018-19,401070216,Phoenix Suns,Isaiah Canaan,4412182,F,True,False,34,8.0,...,0-0,6.0,7.0,1.0,0.0,0.0,0.0,6.0,4.0,-1.0
94331,2018-19,401070216,Phoenix Suns,Isaiah Canaan,2490589,G,True,False,34,8.0,...,0-0,6.0,7.0,1.0,0.0,0.0,0.0,6.0,4.0,7.0
111576,2018-19,401070239,Philadelphia 76ers,Corey Brewer,3191,F,True,False,33.0,11.0,...,1-2,3.0,1.0,3.0,2.0,2.0,0.0,3.0,3.0,17.0
111578,2018-19,401070239,Philadelphia 76ers,Corey Brewer,4415554,F,True,False,33.0,11.0,...,1-2,3.0,1.0,3.0,2.0,2.0,0.0,3.0,3.0,-1.0
94870,2018-19,401070716,Phoenix Suns,Isaiah Canaan,4412182,F,True,False,30,15.0,...,2-3,4.0,2.0,2.0,1.0,0.0,2.0,2.0,3.0,-1.0
94873,2018-19,401070716,Phoenix Suns,Isaiah Canaan,2490589,G,True,False,30,15.0,...,2-3,4.0,2.0,2.0,1.0,0.0,2.0,2.0,3.0,-10.0
95231,2018-19,401070728,Phoenix Suns,Isaiah Canaan,4412182,F,True,False,26,7.0,...,0-0,1.0,2.0,3.0,2.0,0.0,1.0,0.0,1.0,-1.0
95234,2018-19,401070728,Phoenix Suns,Isaiah Canaan,2490589,G,True,False,26,7.0,...,0-0,1.0,2.0,3.0,2.0,0.0,1.0,0.0,1.0,-25.0
95565,2018-19,401070741,Phoenix Suns,Isaiah Canaan,4412182,F,True,False,28,4.0,...,0-0,5.0,2.0,3.0,0.0,0.0,0.0,5.0,2.0,-1.0
95568,2018-19,401070741,Phoenix Suns,Isaiah Canaan,2490589,G,True,False,28,4.0,...,0-0,5.0,2.0,3.0,0.0,0.0,0.0,5.0,2.0,-5.0


### 2.6.7 Locate the fields responsible for identity conflicts

Count the duplicate groups that disagree on player IDs, position, plus/minus, minutes, points, and other statistics.

In [ ]:

# Count repeated-name groups where each variable differs.
# This helps distinguish ID/position inconsistencies
# from genuine statistical disagreements.
check_columns = [
    "PLAYER_ID", "POSITION", "STARTER",
    "DID_NOT_PLAY", "MIN", "PTS", "FG",
    "3PT", "FT", "REB", "AST", "TO",
    "STL", "BLK", "OREB", "DREB", "PF", "+/-"
]

differences = (
    potential_duplicates.groupby(name_keys)[check_columns]
    .nunique(dropna=False)
    .gt(1)
)

difference_summary = (
    differences.sum()
    .sort_values(ascending=False)
    .to_frame("Conflicting Groups")
)

display(difference_summary)


,Conflicting Groups
PLAYER_ID,36
+/-,36
POSITION,28
MIN,7
3PT,7
FG,7
REB,5
PF,5
DREB,5
PTS,5


### 2.6.8 Compare normalized performance statistics

**Methodology:** Convert counting and shooting statistics to comparable numeric types, then distinguish differences in `+/-` from differences in other performance statistics. This reduces false disagreements due solely to inconsistent number formatting.

In [ ]:
import pandas as pd

# Work with the repeated-name records identified earlier.
duplicates_check = potential_duplicates.copy()

name_keys = ["GAME_ID", "TEAM_ID", "PLAYER"]

# Normalize counting statistics before comparing records.
numeric_stats = [
    "PTS", "REB", "AST", "TO", "STL",
    "BLK", "OREB", "DREB", "PF", "+/-"
]

for col in numeric_stats:
    duplicates_check[col] = pd.to_numeric(
        duplicates_check[col], errors="coerce"
    )

# Convert shooting statistics into comparable numerical values.
for col in ["FG", "3PT", "FT"]:
    parts = duplicates_check[col].astype(str).str.split(
        "-", n=1, expand=True
    )

    duplicates_check[f"{col}_MADE"] = pd.to_numeric(
        parts[0], errors="coerce"
    )

    duplicates_check[f"{col}_ATTEMPTED"] = pd.to_numeric(
        parts[1], errors="coerce"
    )

# Normalize playing time.
duplicates_check["MIN_CHECK"] = pd.to_numeric(
    duplicates_check["MIN"], errors="coerce"
)

comparison_stats = (
    numeric_stats
    + ["MIN_CHECK"]
    + [
        f"{col}_{suffix}"
        for col in ["FG", "3PT", "FT"]
        for suffix in ["MADE", "ATTEMPTED"]
    ]
)

# Determine whether each repeated-name group contains
# genuinely different statistical values.
group_differences = (
    duplicates_check.groupby(name_keys)[comparison_stats]
    .nunique(dropna=False)
    .gt(1)
)

classification = pd.DataFrame({
    "DIFFERENT_STATS": group_differences.any(axis=1),
    "DIFFERENT_PLUS_MINUS": group_differences["+/-"]
})

classification["OTHER_STATS_DIFFER"] = (
    group_differences.drop(columns=["+/-"]).any(axis=1)
)

print("Repeated-name groups:", len(classification))

print("\nClassification:")
display(
    classification[
        ["DIFFERENT_PLUS_MINUS", "OTHER_STATS_DIFFER"]
    ].value_counts().to_frame("Groups")
)

print("\nGroups with different statistics excluding +/-:")
display(
    classification.loc[
        classification["OTHER_STATS_DIFFER"]
    ].reset_index()
)

Repeated-name groups: 36

Classification:


Groups
DIFFERENT_PLUS_MINUS OTHER_STATS_DIFFER        
True                 False                   29
                     True                     7


Groups with different statistics excluding +/-:


,GAME_ID,TEAM_ID,PLAYER,DIFFERENT_STATS,DIFFERENT_PLUS_MINUS,OTHER_STATS_DIFFER
0,401070857,21,Isaiah Canaan,True,True,True
1,401070938,21,Isaiah Canaan,True,True,True
2,401071465,17,Mitchell Creek,True,True,True
3,401071551,18,John Jenkins,True,True,True
4,401071587,17,Tahjere McCall,True,True,True
5,401071620,18,Henry Ellenson,True,True,True
6,401160820,14,Daryl Macon,True,True,True


**Normalized comparison:** In 29 groups, plus/minus was the only differing normalized performance statistic. In seven groups, additional minutes, shooting, or counting-statistic differences warranted closer inspection.

### 2.6.9 Review historical player-ID usage

Examine how often each ESPN player ID appears for affected player names across the dataset. Frequency helps prioritize investigation but does **not** independently establish which ID is correct.

In [ ]:

# Find the different ESPN player IDs associated with each
# repeated name, and count how often each ID appears.
identity_summary = (
    data.groupby(["PLAYER", "PLAYER_ID"], dropna=False)
    .agg(
        RECORDS=("GAME_ID", "size"),
        GAMES=("GAME_ID", "nunique"),
        FIRST_SEASON=("SEASON", "min"),
        LAST_SEASON=("SEASON", "max")
    )
    .reset_index()
)

affected_names = (
    potential_duplicates["PLAYER"]
    .dropna()
    .unique()
)

identity_check = identity_summary.loc[
    identity_summary["PLAYER"].isin(affected_names)
].sort_values(
    ["PLAYER", "GAMES"],
    ascending=[True, False]
)

display(identity_check)

# Show the repeated-name records and their IDs.
print("\nRepeated-name records:")
display(
    potential_duplicates[
        [
            "SEASON", "GAME_ID", "TEAM",
            "PLAYER", "PLAYER_ID", "POSITION",
            "MIN", "PTS", "REB", "AST", "+/-"
        ]
    ].sort_values(
        ["PLAYER", "GAME_ID", "PLAYER_ID"]
    )
)


,PLAYER,PLAYER_ID,RECORDS,GAMES,FIRST_SEASON,LAST_SEASON
254,Corey Brewer,3191,287,287,2015-16,2019-20
255,Corey Brewer,4415554,8,8,2018-19,2018-19
309,Daryl Macon,4066243,33,33,2018-19,2019-20
310,Daryl Macon,4610145,4,4,2019-20,2019-20
506,Henry Ellenson,3906522,180,180,2016-17,2020-21
507,Henry Ellenson,4417428,1,1,2018-19,2018-19
524,Isaiah Canaan,2490589,209,209,2015-16,2018-19
525,Isaiah Canaan,4412182,20,20,2018-19,2018-19
718,John Jenkins,6594,116,116,2015-16,2018-19
717,John Jenkins,4415789,1,1,2018-19,2018-19



Repeated-name records:


,SEASON,GAME_ID,TEAM,PLAYER,PLAYER_ID,POSITION,MIN,PTS,REB,AST,+/-
111576,2018-19,401070239,Philadelphia 76ers,Corey Brewer,3191,F,33.0,11.0,3.0,1.0,17.0
111578,2018-19,401070239,Philadelphia 76ers,Corey Brewer,4415554,F,33.0,11.0,3.0,1.0,-1.0
110404,2018-19,401071328,Philadelphia 76ers,Corey Brewer,3191,F,8.0,3.0,2.0,2.0,6.0
110405,2018-19,401071328,Philadelphia 76ers,Corey Brewer,4415554,F,8.0,3.0,2.0,2.0,-1.0
111867,2018-19,401071384,Philadelphia 76ers,Corey Brewer,3191,F,28,9.0,1.0,2.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...
116039,2018-19,401071551,New York Knicks,John Jenkins,6594,G,9,0.0,1.0,0.0,-1.0
113903,2018-19,401071465,Brooklyn Nets,Mitchell Creek,4236300,G,15,8.0,5.0,4.0,-1.0
113898,2018-19,401071465,Brooklyn Nets,Mitchell Creek,4412474,F,--,0.0,0.0,0.0,0.0
116911,2018-19,401071587,Brooklyn Nets,Tahjere McCall,2992257,G,8,4.0,1.0,0.0,-1.0


### 2.6.10 Inspect seven groups with non-plus/minus disagreements

Display both records for each repeated-name group where minutes, scoring, shooting, or other performance values differ.

In [ ]:

import pandas as pd

# Identify repeated player names in the same team-game.
group_keys = ["GAME_ID", "TEAM_ID", "PLAYER"]

duplicates = data.loc[
    data.duplicated(subset=group_keys, keep=False)
].copy()

# Convert statistics to comparable numerical values.
numeric_columns = [
    "MIN", "PTS", "REB", "AST", "TO",
    "STL", "BLK", "OREB", "DREB", "PF", "+/-"
]

for col in numeric_columns:
    duplicates[f"{col}_CHECK"] = pd.to_numeric(
        duplicates[col],
        errors="coerce"
    )

# Separate made and attempted shots.
for col in ["FG", "3PT", "FT"]:
    shots = duplicates[col].astype(str).str.split(
        "-", n=1, expand=True
    )

    duplicates[f"{col}_MADE"] = pd.to_numeric(
        shots[0], errors="coerce"
    )

    duplicates[f"{col}_ATTEMPTED"] = pd.to_numeric(
        shots[1], errors="coerce"
    )

# Compare all performance variables except plus/minus.
performance_columns = (
    [f"{col}_CHECK" for col in numeric_columns if col != "+/-"]
    + [
        f"{col}_{suffix}"
        for col in ["FG", "3PT", "FT"]
        for suffix in ["MADE", "ATTEMPTED"]
    ]
)

# Identify duplicate-name groups with conflicting performance.
conflicting_groups = (
    duplicates.groupby(group_keys)[performance_columns]
    .nunique(dropna=False)
    .gt(1)
    .any(axis=1)
)

conflicting_keys = (
    conflicting_groups[conflicting_groups]
    .index.to_frame(index=False)
)

# Retrieve the original records for detailed inspection.
conflicting_records = duplicates.merge(
    conflicting_keys,
    on=group_keys,
    how="inner"
)

columns_to_display = [
    "SEASON", "GAME_ID", "TEAM", "PLAYER",
    "PLAYER_ID", "POSITION", "STARTER",
    "MIN", "PTS", "FG", "3PT", "FT",
    "REB", "AST", "TO", "STL",
    "BLK", "OREB", "DREB", "PF", "+/-"
]

print("Conflicting groups:", len(conflicting_keys))

display(
    conflicting_records[columns_to_display]
    .sort_values(["GAME_ID", "TEAM", "PLAYER", "PLAYER_ID"])
    .reset_index(drop=True)
)


Conflicting groups: 7


,SEASON,GAME_ID,TEAM,PLAYER,PLAYER_ID,POSITION,STARTER,MIN,PTS,FG,...,FT,REB,AST,TO,STL,BLK,OREB,DREB,PF,+/-
0,2018-19,401070857,Phoenix Suns,Isaiah Canaan,2490589,G,True,26,5.0,2-7,...,0-0,3.0,6.0,2.0,1.0,0.0,1.0,2.0,2.0,-24.0
1,2018-19,401070857,Phoenix Suns,Isaiah Canaan,4412182,F,True,--,0.0,0-0,...,0-0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2018-19,401070938,Phoenix Suns,Isaiah Canaan,2490589,G,True,26,6.0,2-7,...,0-0,0.0,6.0,3.0,0.0,0.0,0.0,0.0,2.0,-4.0
3,2018-19,401070938,Phoenix Suns,Isaiah Canaan,4412182,F,True,--,0.0,0-0,...,0-0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2018-19,401071465,Brooklyn Nets,Mitchell Creek,4236300,G,False,15,8.0,3-5,...,2-3,5.0,4.0,0.0,0.0,0.0,4.0,1.0,1.0,-1.0
5,2018-19,401071465,Brooklyn Nets,Mitchell Creek,4412474,F,False,--,0.0,0-0,...,0-0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,2018-19,401071551,New York Knicks,John Jenkins,4415789,F,False,--,0.0,0-0,...,0-0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,2018-19,401071551,New York Knicks,John Jenkins,6594,G,False,9,0.0,0-3,...,0-0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-1.0
8,2018-19,401071587,Brooklyn Nets,Tahjere McCall,2992257,G,False,8,4.0,2-3,...,0-1,1.0,0.0,0.0,0.0,0.0,0.0,1.0,2.0,-1.0
9,2018-19,401071587,Brooklyn Nets,Tahjere McCall,4416645,F,False,--,0.0,0-0,...,0-0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### 2.6.11 Audit the other 29 repeated-name groups

Check whether those paired records have missing playing time or all-zero counting and shooting statistics. These pairs are not assumed to be zero-performance games merely because their non-plus/minus statistics match.

In [ ]:
import pandas as pd

group_keys = ["GAME_ID", "TEAM_ID", "PLAYER"]

# Select the 29 groups whose performance statistics
# are identical except for plus/minus.
groups_29 = (
    classification.loc[
        ~classification["OTHER_STATS_DIFFER"]
    ]
    .reset_index()[group_keys]
)

records_29 = data.merge(
    groups_29,
    on=group_keys,
    how="inner"
).copy()

# Check whether minutes are unavailable.
records_29["MIN_MISSING"] = (
    pd.to_numeric(
        records_29["MIN"],
        errors="coerce"
    ).isna()
)

# Convert counting statistics to numeric values.
counting_stats = [
    "PTS", "REB", "AST", "TO", "STL",
    "BLK", "OREB", "DREB", "PF"
]

numeric_values = records_29[counting_stats].apply(
    pd.to_numeric,
    errors="coerce"
)

# Verify that every recorded counting statistic is zero.
zero_counting_stats = numeric_values.eq(0).all(axis=1)

# Verify zero shooting makes and attempts.
zero_shooting_stats = (
    records_29[["FG", "3PT", "FT"]]
    .astype(str)
    .apply(lambda col: col.str.strip().eq("0-0"))
    .all(axis=1)
)

records_29["ALL_ZERO_STATS"] = (
    zero_counting_stats & zero_shooting_stats
)

# Summarize both records within each repeated-name group.
group_summary = (
    records_29.groupby(group_keys)
    .agg(
        RECORDS=("PLAYER_ID", "size"),
        MISSING_MINUTES=("MIN_MISSING", "sum"),
        ZERO_STAT_RECORDS=("ALL_ZERO_STATS", "sum")
    )
    .reset_index()
)

print("Repeated-name groups:", len(group_summary))

print("\nBoth records have zero statistics:")
print(
    (
        (group_summary["RECORDS"] == 2)
        & (group_summary["ZERO_STAT_RECORDS"] == 2)
    ).sum()
)

print("\nBoth records have missing minutes:")
print(
    (
        (group_summary["RECORDS"] == 2)
        & (group_summary["MISSING_MINUTES"] == 2)
    ).sum()
)

print("\nDetailed player records:")
display(
    records_29[
        [
            "SEASON", "GAME_ID", "TEAM",
            "PLAYER", "PLAYER_ID", "STARTER",
            "MIN", "PTS", "REB", "AST", "+/-",
            "MIN_MISSING", "ALL_ZERO_STATS"
        ]
    ].sort_values(
        ["GAME_ID", "TEAM", "PLAYER", "PLAYER_ID"]
    )
)


Repeated-name groups: 29

Both records have zero statistics:
2

Both records have missing minutes:
0

Detailed player records:


,SEASON,GAME_ID,TEAM,PLAYER,PLAYER_ID,STARTER,MIN,PTS,REB,AST,+/-,MIN_MISSING,ALL_ZERO_STATS
1,2018-19,401070216,Phoenix Suns,Isaiah Canaan,2490589,True,34,8.0,6.0,7.0,7.0,False,False
0,2018-19,401070216,Phoenix Suns,Isaiah Canaan,4412182,True,34,8.0,6.0,7.0,-1.0,False,False
36,2018-19,401070239,Philadelphia 76ers,Corey Brewer,3191,True,33.0,11.0,3.0,1.0,17.0,False,False
37,2018-19,401070239,Philadelphia 76ers,Corey Brewer,4415554,True,33.0,11.0,3.0,1.0,-1.0,False,False
3,2018-19,401070716,Phoenix Suns,Isaiah Canaan,2490589,True,30,15.0,4.0,2.0,-10.0,False,False
2,2018-19,401070716,Phoenix Suns,Isaiah Canaan,4412182,True,30,15.0,4.0,2.0,-1.0,False,False
5,2018-19,401070728,Phoenix Suns,Isaiah Canaan,2490589,True,26,7.0,1.0,2.0,-25.0,False,False
4,2018-19,401070728,Phoenix Suns,Isaiah Canaan,4412182,True,26,7.0,1.0,2.0,-1.0,False,False
7,2018-19,401070741,Phoenix Suns,Isaiah Canaan,2490589,True,28,4.0,5.0,2.0,-5.0,False,False
6,2018-19,401070741,Phoenix Suns,Isaiah Canaan,4412182,True,28,4.0,5.0,2.0,-1.0,False,False


## 2.7 Controlled identity reconciliation

### 2.7.1 Initial player-name and ID matching audit

**Objective:** Match each repeated-name group to a documented candidate ESPN primary/secondary ID pair in the **same** team-game. This diagnostic records mismatches without deleting data.

**Historical issue:** The candidate dictionary originally used `Mitchel Creek`, whereas the dataset uses `Mitchell Creek`, leaving one pair unmatched. The original diagnostic is preserved.

In [ ]:

import pandas as pd

# Start with the original dataset.
data_cleaned = data.copy()

# Standardize identifiers to avoid mismatches caused
# by text formatting, whitespace, or trailing ".0".
for col in ["GAME_ID", "TEAM_ID", "PLAYER_ID"]:
    data_cleaned[col] = (
        data_cleaned[col]
        .astype(str)
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )

# Previously identified candidate ESPN player-ID pairs.
player_id_mapping = {
    "Corey Brewer": ("3191", "4415554"),
    "Daryl Macon": ("4066243", "4610145"),
    "Henry Ellenson": ("3906522", "4417428"),
    "Isaiah Canaan": ("2490589", "4412182"),
    "John Jenkins": ("6594", "4415789"),
    "Mitchel Creek": ("4236300", "4412474"),
    "Tahjere McCall": ("2992257", "4416645")
}

# Identify repeated names within the same team-game.
group_keys = ["GAME_ID", "TEAM_ID", "PLAYER"]

duplicates = data_cleaned.loc[
    data_cleaned.duplicated(group_keys, keep=False)
].copy()

remove_indices = []
audit_results = []

# Examine each repeated-name group independently.
for (game_id, team_id, player), group in duplicates.groupby(
    group_keys
):
    actual_ids = set(group["PLAYER_ID"])

    expected_ids = player_id_mapping.get(player)

    # Only approve a duplicate group when:
    # 1. Exactly two records exist.
    # 2. Both expected ESPN player IDs are present.
    valid_pair = (
        expected_ids is not None
        and len(group) == 2
        and actual_ids == set(expected_ids)
    )

    audit_results.append({
        "GAME_ID": game_id,
        "TEAM_ID": team_id,
        "PLAYER": player,
        "RECORDS": len(group),
        "PLAYER_IDS": ", ".join(sorted(actual_ids)),
        "VALID_PAIR": valid_pair
    })

    # Flag the secondary player ID only for valid pairs.
    if valid_pair:
        secondary_id = expected_ids[1]

        remove_indices.extend(
            group.loc[
                group["PLAYER_ID"] == secondary_id
            ].index.tolist()
        )

# Review the results before making any changes.
audit_df = pd.DataFrame(audit_results)

print("Repeated-name groups:", len(audit_df))
print("Valid ID pairs:", audit_df["VALID_PAIR"].sum())
print("Records flagged for removal:", len(remove_indices))

print("\nGroups with unmatched player IDs:")
display(
    audit_df.loc[~audit_df["VALID_PAIR"]]
)

print("\nDuplicate matching summary:")
display(
    audit_df.groupby(["PLAYER", "VALID_PAIR"])
    .size()
    .to_frame("Groups")
)


Repeated-name groups: 36
Valid ID pairs: 35
Records flagged for removal: 35

Groups with unmatched player IDs:


,GAME_ID,TEAM_ID,PLAYER,RECORDS,PLAYER_IDS,VALID_PAIR
27,401071465,17,Mitchell Creek,2,"4236300, 4412474",False



Duplicate matching summary:


,,Groups
PLAYER,VALID_PAIR,
Corey Brewer,True,8
Daryl Macon,True,4
Henry Ellenson,True,1
Isaiah Canaan,True,20
John Jenkins,True,1
Mitchell Creek,False,1
Tahjere McCall,True,1


**Preliminary audit:** 35 of the 36 candidate ID pairs matched, with the remaining unmatched name due to the `Mitchel`/`Mitchell` spelling discrepancy. No deletion was performed by this diagnostic cell.

### 2.7.2 Review the unmatched candidate and count flagged rows

**Execution-order adjustment:** This diagnostic originally occurred *before* `remove_indices` was initialized. It has been moved immediately after the matching-audit cell, with its Python contents and saved output unchanged.

In [ ]:

print("Expected duplicate groups:", 36)
print("Records flagged for removal:", len(remove_indices))

print("\nFlagged records by player:")
display(
    data.loc[remove_indices, "PLAYER"]
    .value_counts()
    .to_frame("Flagged Records")
)

print("\nAll repeated-name groups by player:")
display(
    potential_duplicates.groupby("PLAYER")
    .agg(
        RECORDS=("GAME_ID", "size"),
        GAMES=("GAME_ID", "nunique")
    )
)


Expected duplicate groups: 36
Records flagged for removal: 35

Flagged records by player:


,Flagged Records
PLAYER,
Isaiah Canaan,20
Corey Brewer,8
Daryl Macon,4
Henry Ellenson,1
John Jenkins,1
Tahjere McCall,1



All repeated-name groups by player:


,RECORDS,GAMES
PLAYER,,
Corey Brewer,16,8
Daryl Macon,8,4
Henry Ellenson,2,1
Isaiah Canaan,40,20
John Jenkins,2,1
Mitchell Creek,2,1
Tahjere McCall,2,1


### 2.7.3 Correct the candidate mapping's player-name spelling

The following original cell records the spelling correction for **Mitchell Creek**. The subsequent final deletion routine uses player IDs directly, reducing sensitivity to name spelling.

In [ ]:

# Correct the spelling of Mitchell Creek in the mapping.
player_id_mapping["Mitchell Creek"] = (
    "4236300", "4412474"
)

# Remove the incorrectly spelled dictionary key.
player_id_mapping.pop("Mitchel Creek", None)

print("Updated mapping:")
print(player_id_mapping["Mitchell Creek"])


Updated mapping:
('4236300', '4412474')


### 2.7.4 Final ID-pair validation and controlled removal

**Decision rule:** Delete a secondary-ID row only when its verified candidate primary ID is present in the same game and team, and all 36 candidate groups satisfy the audit checks. Store flagged records in `removed_records`.

**Caution:** These are documented, controlled identity decisions; the original unmodified dataset remains the audit reference.

In [ ]:
import pandas as pd

# Always start from the original dataset.
# This prevents accidentally deleting records twice.
data_cleaned = data.copy().reset_index(drop=True)

# Standardize identifiers for consistent matching.
for col in ["GAME_ID", "TEAM_ID", "PLAYER_ID"]:
    data_cleaned[col] = (
        data_cleaned[col]
        .astype(str)
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )

# Map suspected duplicate ESPN IDs to the established IDs.
# These mappings are provisional and require validation.
secondary_to_primary = {
    "4415554": "3191",      # Corey Brewer
    "4610145": "4066243",   # Daryl Macon
    "4417428": "3906522",   # Henry Ellenson
    "4412182": "2490589",   # Isaiah Canaan
    "4415789": "6594",      # John Jenkins
    "4412474": "4236300",   # Mitchell Creek
    "4416645": "2992257"    # Tahjere McCall
}

# Identify players with repeated names in the same team-game.
group_keys = ["GAME_ID", "TEAM_ID", "PLAYER"]

duplicates = data_cleaned.loc[
    data_cleaned.duplicated(group_keys, keep=False)
].copy()

remove_indices = []
audit_results = []

# Examine each repeated-name group and verify its ID pair.
for (game_id, team_id, player), group in duplicates.groupby(
    group_keys, dropna=False
):

    player_ids = set(group["PLAYER_ID"])

    # Find any suspected secondary ID in this group.
    secondary_rows = group.loc[
        group["PLAYER_ID"].isin(secondary_to_primary)
    ]

    valid_pair = False

    if len(group) == 2 and len(secondary_rows) == 1:

        secondary_id = secondary_rows.iloc[0]["PLAYER_ID"]
        primary_id = secondary_to_primary[secondary_id]

        # Only accept the pair when the established ID
        # is present for the same name, team, and game.
        valid_pair = primary_id in player_ids

        if valid_pair:
            remove_indices.append(secondary_rows.index[0])

    audit_results.append({
        "GAME_ID": game_id,
        "TEAM_ID": team_id,
        "PLAYER": player,
        "PLAYER_IDS": ", ".join(sorted(player_ids)),
        "VALID_PAIR": valid_pair
    })

audit_df = pd.DataFrame(audit_results)

print("DUPLICATE VALIDATION")
print("=" * 50)

print("Repeated-name groups:", len(audit_df))
print("Valid ID pairs:", int(audit_df["VALID_PAIR"].sum()))
print("Records flagged for removal:", len(remove_indices))

print("\nUnmatched groups:")
display(audit_df.loc[~audit_df["VALID_PAIR"]])

# Stop if any group does not match the approved candidate IDs.
assert (
    len(audit_df) == 36
    and audit_df["VALID_PAIR"].all()
    and len(remove_indices) == 36
    and len(set(remove_indices)) == 36
), "Duplicate validation failed. Review unmatched groups."

# Preserve an audit trail of the records being removed.
removed_records = data_cleaned.loc[remove_indices].copy()

# Remove only the suspected duplicate identity records.
data_cleaned = (
    data_cleaned.drop(index=remove_indices)
    .reset_index(drop=True)
)

print("\nCLEANING SUMMARY")
print("=" * 50)

print("Original records:", len(data))
print("Removed records:", len(removed_records))
print("Cleaned records:", len(data_cleaned))

print(
    "Remaining repeated-name records:",
    data_cleaned.duplicated(group_keys, keep=False).sum()
)


DUPLICATE VALIDATION
Repeated-name groups: 36
Valid ID pairs: 36
Records flagged for removal: 36

Unmatched groups:


,GAME_ID,TEAM_ID,PLAYER,PLAYER_IDS,VALID_PAIR



CLEANING SUMMARY
Original records: 308263
Removed records: 36
Cleaned records: 308227
Remaining repeated-name records: 0


**Identity-cleaning outcome:** 36 suspected duplicate identity rows removed (308,263 → 308,227). The final repeated-name audit showed zero repeated-name rows under the examined team-game-name key.

## 2.8 Starting-lineup validation after identity correction

Recount reported starters after deduplicating the confirmed candidate pairs. This check is needed because duplicate starter entries can make an otherwise normal team appear to have six or more starters.

In [ ]:

# Standardize the starting-five indicator.
data_cleaned["STARTER_FLAG"] = (
    data_cleaned["STARTER"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("true")
)

# Count starters for every team in every game.
starter_validation = (
    data_cleaned.groupby(
        ["SEASON", "GAME_ID", "TEAM"]
    )["STARTER_FLAG"]
    .sum()
    .reset_index(name="STARTER_COUNT")
)

print("STARTING-FIVE VALIDATION")
print("=" * 50)

print("\nStarter count distribution:")
display(
    starter_validation["STARTER_COUNT"]
    .value_counts()
    .sort_index()
    .to_frame("Team-Games")
)

# Identify team-games without exactly five starters.
invalid_starters = starter_validation.loc[
    starter_validation["STARTER_COUNT"] != 5
]

print("\nTeam-games without exactly five starters:")
print(len(invalid_starters))

display(invalid_starters.head(20))


STARTING-FIVE VALIDATION

Starter count distribution:


,Team-Games
STARTER_COUNT,
5,23955
6,2
8,1



Team-games without exactly five starters:
3


,SEASON,GAME_ID,TEAM,STARTER_COUNT
1873,2015-16,400828826,San Antonio Spurs,6
5473,2017-18,400975023,Minnesota Timberwolves,8
7866,2018-19,401070917,New York Knicks,6


### 2.8.1 Recalculate invalid lineups using team IDs

Use `(SEASON, GAME_ID, TEAM_ID, TEAM)` for precise team-game identification. This replaces a prior name-only inspection with a more reliable group key while preserving the original Python code.

In [ ]:

# Standardize identifiers and starter indicators.
data_cleaned["GAME_ID"] = data_cleaned["GAME_ID"].astype(str)
data_cleaned["TEAM_ID"] = data_cleaned["TEAM_ID"].astype(str)

data_cleaned["STARTER_FLAG"] = (
    data_cleaned["STARTER"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("true")
)

# Include TEAM_ID so we can reliably identify each team.
starter_validation = (
    data_cleaned.groupby(
        ["SEASON", "GAME_ID", "TEAM_ID", "TEAM"]
    )["STARTER_FLAG"]
    .sum()
    .reset_index(name="STARTER_COUNT")
)

# Identify team-games without exactly five starters.
invalid_starters = starter_validation.loc[
    starter_validation["STARTER_COUNT"] != 5
].copy()

print("Total team-games:", len(starter_validation))
print("Team-games with invalid starter counts:", len(invalid_starters))

display(invalid_starters)


Total team-games: 23958
Team-games with invalid starter counts: 3


,SEASON,GAME_ID,TEAM_ID,TEAM,STARTER_COUNT
1873,2015-16,400828826,24,San Antonio Spurs,6
5473,2017-18,400975023,16,Minnesota Timberwolves,8
7866,2018-19,401070917,18,New York Knicks,6


### 2.8.2 Inspect the remaining three lineup exceptions

Examine players marked as starters in the Spurs, Timberwolves, and Knicks games, including DNP status and minute availability. A box-score player with missing minutes should not automatically be removed.

In [ ]:

import pandas as pd

# Inspect the reported starters in each invalid team-game.
for _, game in invalid_starters.iterrows():

    game_players = data_cleaned.loc[
        (data_cleaned["GAME_ID"].astype(str) == str(game["GAME_ID"]))
        & (data_cleaned["TEAM_ID"].astype(str) == str(game["TEAM_ID"]))
    ].copy()

    # Convert playing time for inspection.
    # Values such as "--" become NaN.
    game_players["MIN_CHECK"] = pd.to_numeric(
        game_players["MIN"], errors="coerce"
    )

    # Show only players ESPN identified as starters.
    starters = game_players.loc[
        game_players["STARTER_FLAG"]
    ].copy()

    starters = starters.sort_values(
        "MIN_CHECK",
        ascending=False,
        na_position="last"
    )

    print("\n" + "=" * 65)
    print("SEASON:", game["SEASON"])
    print("TEAM:", game["TEAM"])
    print("GAME ID:", game["GAME_ID"])
    print("REPORTED STARTERS:", len(starters))
    print("=" * 65)

    display(
        starters[
            [
                "PLAYER_ID", "PLAYER", "POSITION",
                "STARTER_FLAG", "DID_NOT_PLAY",
                "MIN", "PTS", "REB", "AST", "+/-"
            ]
        ]
    )

    # Check whether the reported starters have missing minutes.
    print(
        "Starters with missing minutes:",
        starters["MIN_CHECK"].isna().sum()
    )

    # Identify repeated player names among reported starters.
    print(
        "Repeated starter names:",
        starters["PLAYER"].duplicated().sum()
    )



SEASON: 2015-16
TEAM: San Antonio Spurs
GAME ID: 400828826
REPORTED STARTERS: 6


,PLAYER_ID,PLAYER,POSITION,STARTER_FLAG,DID_NOT_PLAY,MIN,PTS,REB,AST,+/-
24030,6450,Kawhi Leonard,F,True,False,40,23.0,6.0,2.0,-3.0
24032,2983,LaMarcus Aldridge,C,True,False,38,23.0,12.0,0.0,1.0
24029,3988,Danny Green,F,True,False,28,4.0,3.0,0.0,1.0
24031,215,Tim Duncan,C,True,False,28,9.0,14.0,4.0,-3.0
24033,1015,Tony Parker,G,True,False,25,6.0,1.0,5.0,-9.0
24034,557,Andre Miller,PG,True,True,NaN,NaN,NaN,NaN,NaN


Starters with missing minutes: 1
Repeated starter names: 0

SEASON: 2017-18
TEAM: Minnesota Timberwolves
GAME ID: 400975023
REPORTED STARTERS: 8


,PLAYER_ID,PLAYER,POSITION,STARTER_FLAG,DID_NOT_PLAY,MIN,PTS,REB,AST,+/-
69950,3136195,Karl-Anthony Towns,C,True,False,39,18.0,11.0,3.0,-18.0
69949,3059319,Andrew Wiggins,F,True,False,39,18.0,3.0,2.0,-2.0
69947,3986,Taj Gibson,F,True,False,37,8.0,8.0,1.0,2.0
69948,6430,Jimmy Butler III,F,True,False,35,18.0,6.0,4.0,-11.0
69952,3135046,Tyus Jones,G,True,False,33,8.0,6.0,6.0,7.0
69951,3192,Aaron Brooks,G,True,False,13,3.0,1.0,2.0,-15.0
69953,2531362,Anthony Brown,F,True,True,NaN,NaN,NaN,NaN,NaN
69954,4015,Jeff Teague,PG,True,True,NaN,NaN,NaN,NaN,NaN


Starters with missing minutes: 2
Repeated starter names: 0

SEASON: 2018-19
TEAM: New York Knicks
GAME ID: 401070917
REPORTED STARTERS: 6


,PLAYER_ID,PLAYER,POSITION,STARTER_FLAG,DID_NOT_PLAY,MIN,PTS,REB,AST,+/-
99981,2528210,Tim Hardaway Jr.,G,True,False,36,32.0,2.0,2.0,-24.0
99979,4278075,Kevin Knox II,F,True,False,23,4.0,2.0,0.0,-16.0
99982,3892818,Emmanuel Mudiay,G,True,False,23,12.0,1.0,2.0,-21.0
99983,3907525,Allonzo Trier,G,True,False,13,7.0,2.0,0.0,-11.0
99980,4351852,Mitchell Robinson,C,True,False,9,0.0,2.0,0.0,-3.0
99978,2995706,Mario Hezonja,F,True,False,--,0.0,0.0,0.0,0.0


Starters with missing minutes: 1
Repeated starter names: 0


**Focused lineup exceptions:** Three team-games required review: San Antonio Spurs (`400828826`), Minnesota Timberwolves (`400975023`), and New York Knicks (`401070917`). The objective was to correct derived starter designations without discarding valid player performances.

### 2.8.3 Apply documented starter-flag corrections

The notebook changes the derived `STARTER_FLAG` only, retaining the original `STARTER` field for auditability. Corrections cover Andre Miller, Anthony Brown, Jeff Teague, Tyus Jones, and Mario Hezonja in their specific historical games.

In [ ]:
# Preserve the original ESPN starter indicator.
# Only the corrected STARTER_FLAG column will be updated.

starter_corrections = [
    ("400828826", "24", "557"),      # Andre Miller
    ("400975023", "16", "2531362"),  # Anthony Brown
    ("400975023", "16", "4015"),     # Jeff Teague
    ("400975023", "16", "3135046"),  # Tyus Jones
    ("401070917", "18", "2995706")   # Mario Hezonja
]

correction_log = []

for game_id, team_id, player_id in starter_corrections:

    # Identify exactly one player in the specified team-game.
    mask = (
        data_cleaned["GAME_ID"].astype(str).eq(game_id)
        & data_cleaned["TEAM_ID"].astype(str).eq(team_id)
        & data_cleaned["PLAYER_ID"].astype(str).eq(player_id)
    )

    assert mask.sum() == 1, (
        f"Expected one record for {player_id} in {game_id}"
    )

    player = data_cleaned.loc[mask, "PLAYER"].iloc[0]
    previous_flag = bool(
        data_cleaned.loc[mask, "STARTER_FLAG"].iloc[0]
    )

    # Correct the lineup designation without changing
    # player performance statistics or the original STARTER field.
    data_cleaned.loc[mask, "STARTER_FLAG"] = False

    correction_log.append({
        "GAME_ID": game_id,
        "PLAYER": player,
        "PREVIOUS_FLAG": previous_flag,
        "CORRECTED_FLAG": False
    })

print("STARTER CORRECTION LOG")
display(pd.DataFrame(correction_log))

STARTER CORRECTION LOG


,GAME_ID,PLAYER,PREVIOUS_FLAG,CORRECTED_FLAG
0,400828826,Andre Miller,True,False
1,400975023,Anthony Brown,True,False
2,400975023,Jeff Teague,True,False
3,400975023,Tyus Jones,True,False
4,401070917,Mario Hezonja,True,False


### 2.8.4 Final five-starter consistency test

Recalculate starter counts for every team-game and assert that no exceptions remain. This verifies lineup **count consistency**; it does not independently verify every one of the approximately 24,000 historical lineups.

In [ ]:

# Recalculate the number of starters for every team-game.
starter_validation = (
    data_cleaned.groupby(
        ["SEASON", "GAME_ID", "TEAM_ID", "TEAM"]
    )["STARTER_FLAG"]
    .sum()
    .reset_index(name="STARTER_COUNT")
)

invalid_starters = starter_validation.loc[
    starter_validation["STARTER_COUNT"] != 5
]

print("FINAL STARTING-FIVE VALIDATION")
print("=" * 50)

print("Total team-games:", len(starter_validation))

print(
    "Team-games with exactly five starters:",
    (starter_validation["STARTER_COUNT"] == 5).sum()
)

print(
    "Team-games with invalid starter counts:",
    len(invalid_starters)
)

print("\nStarter count distribution:")
display(
    starter_validation["STARTER_COUNT"]
    .value_counts()
    .sort_index()
    .to_frame("Team-Games")
)

assert invalid_starters.empty, (
    "Some team-games still have invalid starter counts."
)

print("\nStarting-five validation passed!")


FINAL STARTING-FIVE VALIDATION
Total team-games: 23958
Team-games with exactly five starters: 23958
Team-games with invalid starter counts: 0

Starter count distribution:


,Team-Games
STARTER_COUNT,
5,23958



Starting-five validation passed!


**Final count consistency:** All 23,958 team-games in the uploaded notebook's final lineup validation had exactly five `STARTER_FLAG=True` records, with no count exceptions remaining.

## 2.9 Player-score to team-score reconciliation

**Objective:** Validate that the sum of recorded individual player points equals each team's official final score.

**Methodology:** Aggregate `PTS` by `(GAME_ID, TEAM_ID)`, identify the home/away official score using team IDs, and compute

\[
\text{Point difference} = \sum_{p \in \text{team}} \text{PTS}_p - \text{Official team score}.
\]

Differences are investigated rather than automatically imputed or adjusted.

In [ ]:

import pandas as pd
import numpy as np

# Preserve the cleaned player dataset.
validation_data = data_cleaned.copy()

# Standardize IDs to ensure correct team-game matching.
for col in ["GAME_ID", "TEAM_ID", "HOME_TEAM_ID", "AWAY_TEAM_ID"]:
    validation_data[col] = (
        validation_data[col]
        .astype(str)
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )

# Convert player points and final team scores to numeric values.
for col in ["PTS", "HOME_SCORE", "AWAY_SCORE"]:
    validation_data[col] = pd.to_numeric(
        validation_data[col], errors="coerce"
    )

# Calculate total player points for each team in each game.
# min_count=1 prevents an entirely missing group from becoming zero.
team_points = (
    validation_data.groupby(
        ["GAME_ID", "TEAM_ID"], as_index=False
    )["PTS"]
    .agg(lambda x: x.sum(min_count=1))
    .rename(columns={"PTS": "TOTAL_PLAYER_POINTS"})
)

# Extract one official game record per GAME_ID.
game_results = (
    validation_data[
        [
            "GAME_ID", "HOME_TEAM_ID", "AWAY_TEAM_ID",
            "HOME_SCORE", "AWAY_SCORE"
        ]
    ]
    .drop_duplicates(subset="GAME_ID")
)

# Attach game scores to the aggregated player statistics.
team_validation = team_points.merge(
    game_results,
    on="GAME_ID",
    how="left",
    validate="many_to_one"
)

# Determine whether each player team is home or away.
home_mask = (
    team_validation["TEAM_ID"]
    == team_validation["HOME_TEAM_ID"]
)

away_mask = (
    team_validation["TEAM_ID"]
    == team_validation["AWAY_TEAM_ID"]
)

# Match each team's player points to its official final score.
team_validation["OFFICIAL_TEAM_SCORE"] = np.select(
    [home_mask, away_mask],
    [
        team_validation["HOME_SCORE"],
        team_validation["AWAY_SCORE"]
    ],
    default=np.nan
)

# Calculate discrepancy between summed player points
# and the official game score.
team_validation["POINT_DIFFERENCE"] = (
    team_validation["TOTAL_PLAYER_POINTS"]
    - team_validation["OFFICIAL_TEAM_SCORE"]
)

team_validation["SCORE_MATCH"] = (
    team_validation["POINT_DIFFERENCE"].eq(0)
)

# Treat missing scores as validation failures.
team_validation["SCORE_MATCH"] = (
    team_validation["SCORE_MATCH"]
    & team_validation["TOTAL_PLAYER_POINTS"].notna()
    & team_validation["OFFICIAL_TEAM_SCORE"].notna()
)

print("TEAM SCORE VALIDATION")
print("=" * 55)

print("Total team-games:", len(team_validation))
print("Matching team scores:", team_validation["SCORE_MATCH"].sum())
print("Mismatching team scores:", (~team_validation["SCORE_MATCH"]).sum())

print(
    "Match percentage:",
    round(team_validation["SCORE_MATCH"].mean() * 100, 2),
    "%"
)

# Examine games where the player total does not match.
score_discrepancies = team_validation.loc[
    ~team_validation["SCORE_MATCH"]
].copy()

print("\nSCORE DISCREPANCIES")
display(
    score_discrepancies[
        [
            "GAME_ID", "TEAM_ID",
            "TOTAL_PLAYER_POINTS",
            "OFFICIAL_TEAM_SCORE",
            "POINT_DIFFERENCE"
        ]
    ].head(30)
)

# Summarize the size of discrepancies, if any.
print("\nPOINT DIFFERENCE DISTRIBUTION")
display(
    score_discrepancies["POINT_DIFFERENCE"]
    .value_counts(dropna=False)
    .sort_index()
    .to_frame("Team-Games")
)


TEAM SCORE VALIDATION
Total team-games: 23958
Matching team scores: 22981
Mismatching team scores: 977
Match percentage: 95.92 %

SCORE DISCREPANCIES


,GAME_ID,TEAM_ID,TOTAL_PLAYER_POINTS,OFFICIAL_TEAM_SCORE,POINT_DIFFERENCE
2,400827889,4,0.0,97.0,-97.0
3,400827889,5,0.0,95.0,-95.0
4,400827890,3,0.0,95.0,-95.0
5,400827890,9,0.0,111.0,-111.0
10,400827893,17,0.0,100.0,-100.0
11,400827893,4,0.0,115.0,-115.0
28,400827902,22,0.0,112.0,-112.0
29,400827902,3,0.0,94.0,-94.0
48,400827912,4,0.0,94.0,-94.0
49,400827912,8,0.0,98.0,-98.0



POINT DIFFERENCE DISTRIBUTION


,Team-Games
POINT_DIFFERENCE,
-147.0,1
-146.0,1
-144.0,2
-143.0,2
-142.0,1
...,...
-73.0,1
-69.0,2
-3.0,1


**Initial score-validation finding:** 22,981 of 23,958 team-game player-point sums matched official team scores (95.92%); 977 did not. This check revealed an issue not detected by the five-starter count validation.

### 2.9.1 Diagnose magnitude, direction, and season concentration

Examine whether the reported player totals exceed or fall short of the official score, the number of affected team-games in each season, and the largest differences.

In [ ]:

import pandas as pd
import numpy as np

# Create an audit dataset without modifying the cleaned data.
score_audit = team_validation.copy()

# Add season and team information to identify affected records.
game_seasons = (
    data_cleaned[["GAME_ID", "SEASON"]]
    .drop_duplicates(subset="GAME_ID")
)

team_names = (
    data_cleaned[["GAME_ID", "TEAM_ID", "TEAM"]]
    .drop_duplicates(subset=["GAME_ID", "TEAM_ID"])
)

score_audit = score_audit.merge(
    game_seasons,
    on="GAME_ID",
    how="left"
)

score_audit = score_audit.merge(
    team_names,
    on=["GAME_ID", "TEAM_ID"],
    how="left"
)

# Calculate absolute difference to measure discrepancy size.
score_audit["ABS_DIFFERENCE"] = (
    score_audit["POINT_DIFFERENCE"].abs()
)

mismatches = score_audit.loc[
    ~score_audit["SCORE_MATCH"]
].copy()

print("SCORE DISCREPANCY ANALYSIS")
print("=" * 55)

print("Total mismatches:", len(mismatches))

# Check for missing official scores or unmatched team IDs.
print(
    "Missing official team scores:",
    mismatches["OFFICIAL_TEAM_SCORE"].isna().sum()
)

print(
    "Missing aggregated player points:",
    mismatches["TOTAL_PLAYER_POINTS"].isna().sum()
)

# Determine whether player totals exceed or fall short
# of the final team score.
print(
    "Player points higher than official score:",
    (mismatches["POINT_DIFFERENCE"] > 0).sum()
)

print(
    "Player points lower than official score:",
    (mismatches["POINT_DIFFERENCE"] < 0).sum()
)

print("\nPOINT DIFFERENCE DISTRIBUTION")
display(
    mismatches["POINT_DIFFERENCE"]
    .value_counts(dropna=False)
    .sort_index()
    .to_frame("Team-Games")
)

print("\nMISMATCHES BY SEASON")
season_audit = (
    score_audit.groupby("SEASON")
    .agg(
        TOTAL_TEAM_GAMES=("SCORE_MATCH", "size"),
        MATCHED=("SCORE_MATCH", "sum")
    )
)

season_audit["MISMATCHED"] = (
    season_audit["TOTAL_TEAM_GAMES"]
    - season_audit["MATCHED"]
)

season_audit["MISMATCH_PERCENTAGE"] = (
    season_audit["MISMATCHED"]
    / season_audit["TOTAL_TEAM_GAMES"] * 100
).round(2)

display(season_audit)

print("\nTOP 20 LARGEST DISCREPANCIES")
display(
    mismatches.sort_values(
        "ABS_DIFFERENCE", ascending=False
    )[
        [
            "SEASON", "GAME_ID", "TEAM",
            "TOTAL_PLAYER_POINTS",
            "OFFICIAL_TEAM_SCORE",
            "POINT_DIFFERENCE"
        ]
    ].head(20)
)


SCORE DISCREPANCY ANALYSIS
Total mismatches: 977
Missing official team scores: 0
Missing aggregated player points: 0
Player points higher than official score: 1
Player points lower than official score: 976

POINT DIFFERENCE DISTRIBUTION


,Team-Games
POINT_DIFFERENCE,
-147.0,1
-146.0,1
-144.0,2
-143.0,2
-142.0,1
...,...
-73.0,1
-69.0,2
-3.0,1



MISMATCHES BY SEASON


,TOTAL_TEAM_GAMES,MATCHED,MISMATCHED,MISMATCH_PERCENTAGE
SEASON,,,,
2015-16,2460,2137,323,13.13
2016-17,2460,2134,326,13.25
2017-18,2460,2134,326,13.25
2018-19,2460,2460,0,0.00
2019-20,2118,2117,1,0.05
2020-21,2160,2159,1,0.05
2021-22,2460,2460,0,0.00
2022-23,2460,2460,0,0.00
2023-24,2460,2460,0,0.00



TOP 20 LARGEST DISCREPANCIES


,SEASON,GAME_ID,TEAM,TOTAL_PLAYER_POINTS,OFFICIAL_TEAM_SCORE,POINT_DIFFERENCE
781,2015-16,400828278,Detroit Pistons,0.0,147.0,-147.0
5381,2017-18,400974977,Denver Nuggets,0.0,146.0,-146.0
780,2015-16,400828278,Chicago Bulls,0.0,144.0,-144.0
2446,2015-16,400829113,Minnesota Timberwolves,0.0,144.0,-144.0
3832,2016-17,400900066,Brooklyn Nets,0.0,143.0,-143.0
5481,2017-18,400975027,Golden State Warriors,0.0,143.0,-143.0
3964,2016-17,400900132,Atlanta Hawks,0.0,142.0,-142.0
3965,2016-17,400900132,New York Knicks,0.0,139.0,-139.0
6641,2017-18,400975608,New Orleans Pelicans,0.0,139.0,-139.0
6569,2017-18,400975572,New Orleans Pelicans,0.0,138.0,-138.0


### 2.9.2 Summarize shortages in recorded player scoring

Compute missing points as `OFFICIAL_TEAM_SCORE - TOTAL_PLAYER_POINTS`; large shortages approximately equal to an entire team's score motivate investigation of placeholder box scores.

In [ ]:

# Calculate how many points are missing from each team-game.
# Negative differences mean player totals are below official scores.
mismatches["MISSING_POINTS"] = (
    mismatches["OFFICIAL_TEAM_SCORE"]
    - mismatches["TOTAL_PLAYER_POINTS"]
)

print("MISSING POINTS ANALYSIS")
print("=" * 55)

print("\nSummary statistics:")
display(mismatches["MISSING_POINTS"].describe())

# Identify the most common point differences.
print("\nMost common missing-point amounts:")
display(
    mismatches["MISSING_POINTS"]
    .value_counts()
    .sort_values(ascending=False)
    .head(20)
    .to_frame("Team-Games")
)

# Identify games with the largest point shortages.
print("\nTop 20 largest point shortages:")
display(
    mismatches.sort_values(
        "MISSING_POINTS",
        ascending=False
    )[
        [
            "SEASON", "GAME_ID", "TEAM",
            "TOTAL_PLAYER_POINTS",
            "OFFICIAL_TEAM_SCORE",
            "MISSING_POINTS"
        ]
    ].head(20)
)


MISSING POINTS ANALYSIS

Summary statistics:


,MISSING_POINTS
count,977.000000
mean,105.140225
std,13.452474
min,-2.000000
25%,96.000000
50%,105.000000
75%,114.000000
max,147.000000



Most common missing-point amounts:


,Team-Games
MISSING_POINTS,
105.0,41
94.0,34
100.0,34
107.0,33
115.0,33
98.0,29
95.0,28
114.0,28
99.0,28



Top 20 largest point shortages:


,SEASON,GAME_ID,TEAM,TOTAL_PLAYER_POINTS,OFFICIAL_TEAM_SCORE,MISSING_POINTS
781,2015-16,400828278,Detroit Pistons,0.0,147.0,147.0
5381,2017-18,400974977,Denver Nuggets,0.0,146.0,146.0
780,2015-16,400828278,Chicago Bulls,0.0,144.0,144.0
2446,2015-16,400829113,Minnesota Timberwolves,0.0,144.0,144.0
3832,2016-17,400900066,Brooklyn Nets,0.0,143.0,143.0
5481,2017-18,400975027,Golden State Warriors,0.0,143.0,143.0
3964,2016-17,400900132,Atlanta Hawks,0.0,142.0,142.0
3965,2016-17,400900132,New York Knicks,0.0,139.0,139.0
6641,2017-18,400975608,New Orleans Pelicans,0.0,139.0,139.0
6569,2017-18,400975572,New Orleans Pelicans,0.0,138.0,138.0


## 2.10 Identify incomplete and placeholder game box scores

**Objective:** Separate valid zero scoring by a player from **impossible zero player scoring for an entire team** that officially scored points.

**Rule:** Flag a team-game when total recorded player points is zero but the official team score is positive, then count how many teams per game are affected.

In [ ]:
import pandas as pd

# Work from the existing team-score validation results.
score_check = team_validation.copy()

# Identify team-games where the sum of player points is zero
# even though the official team score is positive.
score_check["ZERO_PLAYER_POINTS"] = (
    score_check["TOTAL_PLAYER_POINTS"].eq(0)
    & score_check["OFFICIAL_TEAM_SCORE"].gt(0)
)

# Separate discrepancies into three categories.
score_check["ISSUE_TYPE"] = "MATCHED"

score_check.loc[
    ~score_check["SCORE_MATCH"],
    "ISSUE_TYPE"
] = "OTHER_SCORE_MISMATCH"

score_check.loc[
    score_check["ZERO_PLAYER_POINTS"],
    "ISSUE_TYPE"
] = "ZERO_PLAYER_POINTS"

print("SCORE DISCREPANCY CLASSIFICATION")
print("=" * 55)

display(
    score_check["ISSUE_TYPE"]
    .value_counts()
    .to_frame("Team-Games")
)

# Determine whether both teams in the same game
# have zero recorded player points.
game_check = (
    score_check.groupby("GAME_ID")
    .agg(
        TEAMS=("TEAM_ID", "nunique"),
        ZERO_POINT_TEAMS=("ZERO_PLAYER_POINTS", "sum")
    )
    .reset_index()
)

print("\nZERO-POINT TEAMS PER GAME")
display(
    game_check["ZERO_POINT_TEAMS"]
    .value_counts()
    .sort_index()
    .to_frame("Games")
)

print("\nGames with both teams missing player scoring:")
print((game_check["ZERO_POINT_TEAMS"] == 2).sum())

print("\nGames with only one team missing player scoring:")
print((game_check["ZERO_POINT_TEAMS"] == 1).sum())


SCORE DISCREPANCY CLASSIFICATION


,Team-Games
ISSUE_TYPE,
MATCHED,22981
ZERO_PLAYER_POINTS,974
OTHER_SCORE_MISMATCH,3



ZERO-POINT TEAMS PER GAME


,Games
ZERO_POINT_TEAMS,
0,11492
2,487



Games with both teams missing player scoring:
487

Games with only one team missing player scoring:
0


**Discrepancy classification:** Of 977 mismatched team-games, 974 had **zero recorded player points**, spanning 487 games where *both teams* were affected. Three remaining team-games had nonzero player points but small score discrepancies.

### 2.10.1 Inspect an example placeholder game

Display the player records and original `MIN` values for a team whose player-point total is zero despite a positive official score. This provides direct evidence of placeholder rows rather than genuine player performances.

In [ ]:

# Select a game where an official score exists
# but the sum of player points is zero.
zero_point_games = score_check.loc[
    score_check["ZERO_PLAYER_POINTS"]
]

if zero_point_games.empty:
    print("No zero-point team-games found.")

else:
    sample = zero_point_games.iloc[0]

    game_id = str(sample["GAME_ID"])
    team_id = str(sample["TEAM_ID"])

    # Retrieve the player records from the original
    # cleaned player-game dataset.
    sample_players = data_cleaned.loc[
        data_cleaned["GAME_ID"].astype(str).eq(game_id)
        & data_cleaned["TEAM_ID"].astype(str).eq(team_id)
    ].copy()

    print("GAME ID:", game_id)
    print("TEAM:", sample_players["TEAM"].iloc[0])
    print("OFFICIAL SCORE:", sample["OFFICIAL_TEAM_SCORE"])
    print("TOTAL PLAYER POINTS:", sample["TOTAL_PLAYER_POINTS"])

    print("\nPLAYER RECORDS")
    display(
        sample_players[
            [
                "PLAYER", "STARTER_FLAG", "DID_NOT_PLAY",
                "MIN", "PTS", "FG", "3PT", "FT",
                "REB", "AST"
            ]
        ]
    )

    print("\nOriginal minutes distribution:")
    display(
        sample_players["MIN"]
        .value_counts(dropna=False)
        .to_frame("Records")
    )


GAME ID: 400827889
TEAM: Chicago Bulls
OFFICIAL SCORE: 97.0
TOTAL PLAYER POINTS: 0.0

PLAYER RECORDS


,PLAYER,STARTER_FLAG,DID_NOT_PLAY,MIN,PTS,FG,3PT,FT,REB,AST
26,Pau Gasol,True,False,--,0.0,0-0,0-0,0-0,0.0,0.0
27,Jimmy Butler III,True,False,--,0.0,0-0,0-0,0-0,0.0,0.0
28,Nikola Mirotic,True,False,--,0.0,0-0,0-0,0-0,0.0,0.0
29,Tony Snell,True,False,--,0.0,0-0,0-0,0-0,0.0,0.0
30,Derrick Rose,True,False,--,0.0,0-0,0-0,0-0,0.0,0.0
31,Taj Gibson,False,False,--,0.0,0-0,0-0,0-0,0.0,0.0
32,Doug McDermott,False,False,--,0.0,0-0,0-0,0-0,0.0,0.0
33,Joakim Noah,False,False,--,0.0,0-0,0-0,0-0,0.0,0.0
34,Aaron Brooks,False,False,--,0.0,0-0,0-0,0-0,0.0,0.0
35,E'Twaun Moore,False,False,--,0.0,0-0,0-0,0-0,0.0,0.0



Original minutes distribution:


,Records
MIN,
--,10
NaN,3


### 2.10.2 Analyze placeholder-game availability by season

Compute the number and percentage of games in each season where **both teams** have placeholder player scoring. Preserve those games' final outcomes for team-based modeling while excluding the unreliable player box scores from performance calculations.

In [ ]:

import pandas as pd

# Identify team-games with zero recorded player points
# despite having a positive official final score.
quality = team_validation.copy()

quality["ZERO_PLAYER_POINTS"] = (
    quality["TOTAL_PLAYER_POINTS"].eq(0)
    & quality["OFFICIAL_TEAM_SCORE"].gt(0)
)

# Classify games based on whether both teams
# have zero player points.
game_quality = (
    quality.groupby("GAME_ID")
    .agg(
        TEAM_COUNT=("TEAM_ID", "nunique"),
        ZERO_POINT_TEAMS=("ZERO_PLAYER_POINTS", "sum")
    )
    .reset_index()
)

# A placeholder game has two teams with zero player points.
game_quality["NO_PLAYER_STATS"] = (
    game_quality["TEAM_COUNT"].eq(2)
    & game_quality["ZERO_POINT_TEAMS"].eq(2)
)

# Attach NBA season information to each game.
season_lookup = (
    data_cleaned[["GAME_ID", "SEASON"]]
    .drop_duplicates(subset="GAME_ID")
)

game_quality = game_quality.merge(
    season_lookup,
    on="GAME_ID",
    how="left",
    validate="one_to_one"
)

# Calculate the number and percentage of affected games
# within each NBA season.
season_missing = (
    game_quality.groupby("SEASON")
    .agg(
        TOTAL_GAMES=("GAME_ID", "nunique"),
        GAMES_NO_STATS=("NO_PLAYER_STATS", "sum")
    )
    .reset_index()
    .sort_values("SEASON")
)

season_missing["GAMES_WITH_STATS"] = (
    season_missing["TOTAL_GAMES"]
    - season_missing["GAMES_NO_STATS"]
)

season_missing["MISSING_PERCENTAGE (%)"] = (
    season_missing["GAMES_NO_STATS"]
    / season_missing["TOTAL_GAMES"] * 100
).round(2)

season_missing["AVAILABLE_PERCENTAGE (%)"] = (
    season_missing["GAMES_WITH_STATS"]
    / season_missing["TOTAL_GAMES"] * 100
).round(2)

print("PLAYER BOXSCORE AVAILABILITY BY SEASON")
print("=" * 65)

display(season_missing)

print("\nOVERALL SUMMARY")
print("Total games:", len(game_quality))
print("Games with zero player scoring:",
      game_quality["NO_PLAYER_STATS"].sum())
print(
    "Overall missing percentage:",
    round(game_quality["NO_PLAYER_STATS"].mean() * 100, 2),
    "%"
)

# List affected game IDs for future data recovery.
print("\nGAMES WITH NO PLAYER SCORING BY SEASON")

display(
    game_quality.loc[
        game_quality["NO_PLAYER_STATS"],
        ["SEASON", "GAME_ID"]
    ].sort_values(["SEASON", "GAME_ID"])
)


PLAYER BOXSCORE AVAILABILITY BY SEASON


,SEASON,TOTAL_GAMES,GAMES_NO_STATS,GAMES_WITH_STATS,MISSING_PERCENTAGE (%),AVAILABLE_PERCENTAGE (%)
0,2015-16,1230,161,1069,13.09,86.91
1,2016-17,1230,163,1067,13.25,86.75
2,2017-18,1230,163,1067,13.25,86.75
3,2018-19,1230,0,1230,0.00,100.00
4,2019-20,1059,0,1059,0.00,100.00
5,2020-21,1080,0,1080,0.00,100.00
6,2021-22,1230,0,1230,0.00,100.00
7,2022-23,1230,0,1230,0.00,100.00
8,2023-24,1230,0,1230,0.00,100.00
9,2024-25,1230,0,1230,0.00,100.00



OVERALL SUMMARY
Total games: 11979
Games with zero player scoring: 487
Overall missing percentage: 4.07 %

GAMES WITH NO PLAYER SCORING BY SEASON


,SEASON,GAME_ID
1,2015-16,400827889
2,2015-16,400827890
5,2015-16,400827893
14,2015-16,400827902
24,2015-16,400827912
...,...,...
3663,2017-18,400975951
3670,2017-18,400975958
3682,2017-18,400975970
3684,2017-18,400975972


**Season-level outcome:** Placeholder games were confined to 2015–16 (161 games), 2016–17 (163), and 2017–18 (163). No complete-game zero-player-scoring placeholders were identified for 2018–19 through 2024–25. This does not prove every non-placeholder individual statistic is error-free.

## 2.11 Review small, non-placeholder score differences

Inspect the three team-games whose recorded player scoring is nonzero but differs slightly from the official score. Player-level corrections should be traceable to independently reviewed box scores; raw inputs should remain available.

In [ ]:

# Identify team-games with score discrepancies,
# excluding games with zero recorded player points.
remaining_mismatches = team_validation.loc[
    (~team_validation["SCORE_MATCH"])
    & (team_validation["TOTAL_PLAYER_POINTS"] > 0)
].copy()

# Add season and team details for easier investigation.
team_lookup = (
    data_cleaned[
        ["GAME_ID", "TEAM_ID", "SEASON", "TEAM"]
    ]
    .drop_duplicates(["GAME_ID", "TEAM_ID"])
)

remaining_mismatches = remaining_mismatches.merge(
    team_lookup,
    on=["GAME_ID", "TEAM_ID"],
    how="left"
)

print("REMAINING SCORE MISMATCHES")
print("=" * 55)

display(
    remaining_mismatches[
        [
            "SEASON", "GAME_ID", "TEAM",
            "TOTAL_PLAYER_POINTS",
            "OFFICIAL_TEAM_SCORE",
            "POINT_DIFFERENCE"
        ]
    ]
)

# Examine the actual player records for each affected team.
for _, game in remaining_mismatches.iterrows():

    records = data_cleaned.loc[
        data_cleaned["GAME_ID"].astype(str).eq(
            str(game["GAME_ID"])
        )
        & data_cleaned["TEAM_ID"].astype(str).eq(
            str(game["TEAM_ID"])
        )
    ].copy()

    print("\n" + "=" * 60)
    print("SEASON:", game["SEASON"])
    print("TEAM:", game["TEAM"])
    print("GAME ID:", game["GAME_ID"])
    print("OFFICIAL SCORE:", game["OFFICIAL_TEAM_SCORE"])
    print("PLAYER POINTS:", game["TOTAL_PLAYER_POINTS"])
    print("DIFFERENCE:", game["POINT_DIFFERENCE"])

    display(
        records[
            [
                "PLAYER", "PLAYER_ID", "MIN",
                "PTS", "FG", "3PT", "FT",
                "STARTER_FLAG", "DID_NOT_PLAY"
            ]
        ]
    )


REMAINING SCORE MISMATCHES


,SEASON,GAME_ID,TEAM,TOTAL_PLAYER_POINTS,OFFICIAL_TEAM_SCORE,POINT_DIFFERENCE
0,2015-16,400828881,Cleveland Cavaliers,84.0,85.0,-1.0
1,2019-20,401161182,Brooklyn Nets,91.0,89.0,2.0
2,2020-21,401267209,Philadelphia 76ers,97.0,100.0,-3.0



SEASON: 2015-16
TEAM: Cleveland Cavaliers
GAME ID: 400828881
OFFICIAL SCORE: 85.0
PLAYER POINTS: 84.0
DIFFERENCE: -1.0


,PLAYER,PLAYER_ID,MIN,PTS,FG,3PT,FT,STARTER_FLAG,DID_NOT_PLAY
25440,LeBron James,1966,37,23.0,10-20,2-6,1-2,True,False
25441,Kevin Love,3449,34,12.0,4-11,2-7,2-2,True,False
25442,Timofey Mozgov,4298,16,7.0,3-3,0-0,1-2,True,False
25443,JR Smith,2444,29,5.0,2-12,1-8,0-0,True,False
25444,Kyrie Irving,6442,34,15.0,7-23,1-10,0-0,True,False
25445,Richard Jefferson,1006,9,5.0,2-3,1-2,0-0,False,False
25446,Channing Frye,2754,19,7.0,3-6,1-3,0-0,False,False
25447,Sasha Kaun,3443,--,0.0,0-0,0-0,0-0,False,False
25448,Tristan Thompson,6474,20,2.0,1-1,0-0,0-0,False,False
25449,Iman Shumpert,6468,19,3.0,1-3,1-2,0-0,False,False



SEASON: 2019-20
TEAM: Brooklyn Nets
GAME ID: 401161182
OFFICIAL SCORE: 89.0
PLAYER POINTS: 91.0
DIFFERENCE: 2.0


,PLAYER,PLAYER_ID,MIN,PTS,FG,3PT,FT,STARTER_FLAG,DID_NOT_PLAY
138364,Taurean Prince,2990962,29.0,6.0,2-11,2-9,0-0,True,False
138365,Joe Harris,2528794,31.0,16.0,6-11,3-7,1-1,True,False
138366,Jarrett Allen,4066328,25.0,5.0,2-6,0-0,1-1,True,False
138367,Spencer Dinwiddie,2580782,29.0,16.0,6-19,2-10,2-4,True,False
138368,Garrett Temple,4023,29.0,9.0,2-13,1-8,4-5,True,False
138369,Wilson Chandler,3194,18.0,9.0,3-10,1-6,2-2,False,False
138370,Justin Anderson,2982340,2.0,2.0,1-1,0-0,0-0,False,False
138371,Dzanan Musa,4348696,14.0,6.0,1-4,0-0,4-4,False,False
138372,Rodions Kurucs,4230548,12.0,7.0,3-5,1-2,0-0,False,False
138373,DeAndre Jordan,3442,21.0,4.0,2-3,0-0,0-0,False,False



SEASON: 2020-21
TEAM: Philadelphia 76ers
GAME ID: 401267209
OFFICIAL SCORE: 100.0
PLAYER POINTS: 97.0
DIFFERENCE: -3.0


,PLAYER,PLAYER_ID,MIN,PTS,FG,3PT,FT,STARTER_FLAG,DID_NOT_PLAY
152681,Danny Green,3988,29.0,7.0,3-7,1-4,0-0,True,False
152682,Tobias Harris,6440,38.0,26.0,11-20,2-5,2-2,True,False
152683,Joel Embiid,3059318,38.0,29.0,7-17,1-6,14-16,True,False
152684,Seth Curry,2326307,33.0,17.0,5-11,3-6,4-4,True,False
152685,Ben Simmons,3907387,37.0,11.0,3-11,0-0,5-8,True,False
152686,Dwight Howard,2384,10.0,0.0,0-0,0-0,0-0,False,False
152687,Shake Milton,3915195,26.0,7.0,2-10,1-6,2-2,False,False
152688,Matisse Thybulle,3907498,12.0,0.0,0-2,0-0,0-0,False,False
152689,Tyrese Maxey,4431678,8.0,0.0,0-3,0-2,0-0,False,False
152690,Mike Scott,6622,NaN,NaN,NaN,NaN,NaN,False,True


### 2.11.1 Apply documented source-specific corrections

The notebook preserves `data_cleaned` and makes reviewed changes in `data_final`, recording every changed field in `correction_audit` along with its recorded source URL. The three affected games involve Sasha Kaun, Iman Shumpert, and Mike Scott.

**Note:** Corrections and links are preserved **as recorded in the uploaded notebook**; this documentation pass does not independently reverify external sources.

In [ ]:

import pandas as pd

# Preserve the existing cleaned dataset.
data_final = data_cleaned.copy()

# Standardize identifiers for exact record matching.
for col in ["GAME_ID", "PLAYER_ID"]:
    data_final[col] = data_final[col].astype(str).str.strip()

correction_log = []


def correct_player(game_id, player_id, updates, source):
    """
    Correct one independently verified player-game record.

    Parameters
    ----------
    game_id : str
        ESPN game identifier.
    player_id : str
        ESPN player identifier.
    updates : dict
        Column names and their verified replacement values.
    source : str
        Reference URL supporting the correction.
    """
    mask = (
        data_final["GAME_ID"].eq(game_id)
        & data_final["PLAYER_ID"].eq(player_id)
    )

    assert mask.sum() == 1, (
        f"Expected exactly one player record: "
        f"{game_id}, {player_id}"
    )

    idx = data_final.index[mask][0]
    player = data_final.at[idx, "PLAYER"]

    for column, new_value in updates.items():
        # Record every correction for reproducibility.
        correction_log.append({
            "GAME_ID": game_id,
            "PLAYER_ID": player_id,
            "PLAYER": player,
            "COLUMN": column,
            "OLD_VALUE": data_final.at[idx, column]
                if column in data_final.columns else None,
            "NEW_VALUE": new_value,
            "SOURCE": source
        })

        data_final.at[idx, column] = new_value


# 1. Cleveland Cavaliers: Sasha Kaun made one free throw.
correct_player(
    "400828881",
    "3443",
    {
        "PTS": 1,
        "FT": "1-2"
    },
    "https://www.basketball-reference.com/boxscores/201603140UTA.html"
)

# 2. Philadelphia 76ers: Restore Mike Scott's box score.
# ESPN incorrectly classified his appearance as DNP.
mike_scott_updates = {
    "DID_NOT_PLAY": False,
    "MIN": "10",
    "PTS": 3,
    "FG": "1-3",
    "3PT": "0-2",
    "FT": "1-1",
    "REB": 2,
    "AST": 0,
    "TO": 0,
    "STL": 0,
    "BLK": 1,
    "OREB": 1,
    "DREB": 1,
    "PF": 2,
    "+/-": -14
}

# Update already-derived variables if they exist.
if "DNP_FLAG" in data_final.columns:
    mike_scott_updates["DNP_FLAG"] = False

if "MIN_NUM" in data_final.columns:
    mike_scott_updates["MIN_NUM"] = 10 + 8 / 60

correct_player(
    "401267209",
    "6622",
    mike_scott_updates,
    "https://www.basketball-reference.com/boxscores/202012290PHI.html"
)

# 3. Brooklyn Nets: Flag an incorrect Iman Shumpert record.
shumpert_mask = (
    data_final["GAME_ID"].eq("401161182")
    & data_final["PLAYER_ID"].eq("6468")
)

assert shumpert_mask.sum() == 1, (
    "Review the Brooklyn record before deletion."
)

assert pd.to_numeric(
    data_final.loc[shumpert_mask, "PTS"]
).iloc[0] == 2, "The Brooklyn record differs from the reviewed case."

correction_log.append({
    "GAME_ID": "401161182",
    "PLAYER_ID": "6468",
    "PLAYER": "Iman Shumpert",
    "COLUMN": "ROW",
    "OLD_VALUE": "Present: 2 points",
    "NEW_VALUE": "Removed: absent from official box score",
    "SOURCE": (
        "https://www.nba.com/nets/game/"
        "0021900540-nets-vs-magic-orlando-fl-01-06-2020"
    )
})

data_final = data_final.loc[
    ~shumpert_mask
].copy()

# Display the complete correction audit.
correction_audit = pd.DataFrame(correction_log)

print("SOURCE-VERIFIED CORRECTIONS")
display(correction_audit)

print("\nOriginal cleaned records:", len(data_cleaned))
print("Final records:", len(data_final))


SOURCE-VERIFIED CORRECTIONS


,GAME_ID,PLAYER_ID,PLAYER,COLUMN,OLD_VALUE,NEW_VALUE,SOURCE
0,400828881,3443,Sasha Kaun,PTS,0.0,1,https://www.basketball-reference.com/boxscores...
1,400828881,3443,Sasha Kaun,FT,0-0,1-2,https://www.basketball-reference.com/boxscores...
2,401267209,6622,Mike Scott,DID_NOT_PLAY,True,False,https://www.basketball-reference.com/boxscores...
3,401267209,6622,Mike Scott,MIN,NaN,10,https://www.basketball-reference.com/boxscores...
4,401267209,6622,Mike Scott,PTS,NaN,3,https://www.basketball-reference.com/boxscores...
5,401267209,6622,Mike Scott,FG,NaN,1-3,https://www.basketball-reference.com/boxscores...
6,401267209,6622,Mike Scott,3PT,NaN,0-2,https://www.basketball-reference.com/boxscores...
7,401267209,6622,Mike Scott,FT,NaN,1-1,https://www.basketball-reference.com/boxscores...
8,401267209,6622,Mike Scott,REB,NaN,2,https://www.basketball-reference.com/boxscores...
9,401267209,6622,Mike Scott,AST,NaN,0,https://www.basketball-reference.com/boxscores...



Original cleaned records: 308227
Final records: 308226


### 2.11.2 Recalculate official-score consistency

After correcting or removing the reviewed player records, recompute team point totals and rerun reconciliation across all team-games, including the three targeted cases.

In [ ]:

import numpy as np

# Recalculate team totals because the previous validation
# was based on records before the three corrections.
for col in ["TEAM_ID", "HOME_TEAM_ID", "AWAY_TEAM_ID"]:
    data_final[col] = data_final[col].astype(str).str.strip()

data_final["PTS"] = pd.to_numeric(
    data_final["PTS"], errors="coerce"
)

team_totals = (
    data_final.groupby(["GAME_ID", "TEAM_ID"], as_index=False)
    .agg(PLAYER_POINTS=("PTS", "sum"))
)

game_scores = (
    data_final[
        [
            "GAME_ID", "HOME_TEAM_ID", "AWAY_TEAM_ID",
            "HOME_SCORE", "AWAY_SCORE"
        ]
    ]
    .drop_duplicates("GAME_ID")
)

score_validation = team_totals.merge(
    game_scores,
    on="GAME_ID",
    validate="many_to_one"
)

score_validation["OFFICIAL_POINTS"] = np.select(
    [
        score_validation["TEAM_ID"].eq(
            score_validation["HOME_TEAM_ID"]
        ),
        score_validation["TEAM_ID"].eq(
            score_validation["AWAY_TEAM_ID"]
        )
    ],
    [
        score_validation["HOME_SCORE"],
        score_validation["AWAY_SCORE"]
    ],
    default=np.nan
)

score_validation["SCORE_MATCH"] = (
    score_validation["PLAYER_POINTS"]
    .eq(score_validation["OFFICIAL_POINTS"])
)

# Print the score validation results after corrections.
print("UPDATED SCORE VALIDATION")
print("=" * 50)

print("Total team-games:", len(score_validation))
print("Matching scores:", score_validation["SCORE_MATCH"].sum())
print("Mismatching scores:", (~score_validation["SCORE_MATCH"]).sum())

# Verify the three specific team-games.
reviewed_games = [
    "400828881",
    "401161182",
    "401267209"
]

print("\nVERIFIED GAME RESULTS")
display(
    score_validation.loc[
        score_validation["GAME_ID"].isin(reviewed_games),
        [
            "GAME_ID", "TEAM_ID", "PLAYER_POINTS",
            "OFFICIAL_POINTS", "SCORE_MATCH"
        ]
    ]
)


UPDATED SCORE VALIDATION
Total team-games: 23958
Matching scores: 22984
Mismatching scores: 974

VERIFIED GAME RESULTS


,GAME_ID,TEAM_ID,PLAYER_POINTS,OFFICIAL_POINTS,SCORE_MATCH
1982,400828881,26,94.0,94.0,True
1983,400828881,5,85.0,85.0,True
10918,401161182,17,89.0,89.0,True
10919,401161182,19,101.0,101.0,True
12052,401267209,20,100.0,100.0,True
12053,401267209,28,93.0,93.0,True


**Post-correction score validation:** 22,984 team-games matched and 974 did not. All three additional reviewed score discrepancies were resolved in the notebook's outputs; the 974 residual mismatches represent the two teams in the 487 placeholder games.

### 2.11.3 Inspect the complete reconciliation table

This existing exploratory output displays the team-game score-validation DataFrame for review.

In [ ]:
score_validation

,GAME_ID,TEAM_ID,PLAYER_POINTS,HOME_TEAM_ID,AWAY_TEAM_ID,HOME_SCORE,AWAY_SCORE,OFFICIAL_POINTS,SCORE_MATCH
0,400827888,1,94.0,1,8,94,106,94.0,True
1,400827888,8,106.0,1,8,94,106,106.0,True
2,400827889,4,0.0,4,5,97,95,97.0,False
3,400827889,5,0.0,4,5,97,95,95.0,False
4,400827890,3,0.0,9,3,111,95,95.0,False
...,...,...,...,...,...,...,...,...,...
23953,401748706,30,88.0,12,30,123,88,88.0,True
23954,401754705,13,125.0,13,24,125,109,125.0,True
23955,401754705,24,109.0,13,24,125,109,109.0,True
23956,401754706,15,111.0,3,15,107,111,111.0,True


## 2.12 Final quality classification and export

**Objective:** Produce reproducible datasets that preserve historical outcomes but exclude incomplete game box scores from player-performance averages.

**Data quality classification:**

- `COMPLETE` — both teams' player-point totals match official scores.
- `PLACEHOLDER_BOXSCORE` — both teams have zero player points despite positive official scores.
- `REVIEW_REQUIRED` — any remaining team-game score inconsistency.

The exported **player performance** subset contains only `COMPLETE` games and players not marked DNP. It does **not** remove valid zero-point or zero-rebound games automatically.

**Saved outputs** (under `NBA Dataset/cleaned/`):

| File | Use |
|---|---|
| `nba_games_cleaned.csv` | All 11,979 game outcomes, with quality indicators |
| `nba_players_cleaned.csv` | All retained player records, including DNP and placeholder-game rows |
| `nba_players_performance.csv` | Non-DNP player records from score-validated games |
| `nba_game_quality_report.csv` | Game-level score quality/status flags |
| `nba_correction_audit.csv` | Source-specific corrected fields and removed player row |

In [ ]:

import pandas as pd
from pathlib import Path

BASE_DIR = Path(
    "/content/drive/MyDrive/Colab Notebooks/Dataset/NBA Dataset"
)

OUTPUT_DIR = BASE_DIR / "cleaned"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Work from the updated score validation.
team_quality = score_validation.copy()

team_quality["GAME_ID"] = (
    team_quality["GAME_ID"].astype(str)
)

# Identify placeholder team box scores.
# These teams scored points in the actual game,
# but their player records contain zero total points.
team_quality["PLACEHOLDER_TEAM"] = (
    team_quality["PLAYER_POINTS"].eq(0)
    & team_quality["OFFICIAL_POINTS"].gt(0)
)

# Classify data quality at the game level.
# A game is considered complete only when both teams'
# player point totals match their official scores.
game_quality = (
    team_quality.groupby("GAME_ID")
    .agg(
        TEAM_COUNT=("TEAM_ID", "nunique"),
        MATCHING_TEAMS=("SCORE_MATCH", "sum"),
        PLACEHOLDER_TEAMS=("PLACEHOLDER_TEAM", "sum")
    )
    .reset_index()
)

game_quality["QUALITY_STATUS"] = "REVIEW_REQUIRED"

game_quality.loc[
    game_quality["TEAM_COUNT"].eq(2)
    & game_quality["MATCHING_TEAMS"].eq(2),
    "QUALITY_STATUS"
] = "COMPLETE"

game_quality.loc[
    game_quality["TEAM_COUNT"].eq(2)
    & game_quality["PLACEHOLDER_TEAMS"].eq(2),
    "QUALITY_STATUS"
] = "PLACEHOLDER_BOXSCORE"

game_quality["PLAYER_STATS_VALID"] = (
    game_quality["QUALITY_STATUS"].eq("COMPLETE")
)

print("GAME QUALITY SUMMARY")
display(game_quality["QUALITY_STATUS"].value_counts())

# Stop if new unexplained scoring discrepancies appear.
assert len(game_quality) == 11979
assert (game_quality["QUALITY_STATUS"] == "COMPLETE").sum() == 11492
assert (game_quality["QUALITY_STATUS"] == "PLACEHOLDER_BOXSCORE").sum() == 487


# Attach quality flags to every player record.
players_cleaned = data_final.merge(
    game_quality[
        ["GAME_ID", "QUALITY_STATUS", "PLAYER_STATS_VALID"]
    ],
    on="GAME_ID",
    how="left",
    validate="many_to_one"
)

# Preserve non-participating players in the full dataset.
players_cleaned["DNP_FLAG"] = (
    players_cleaned["DID_NOT_PLAY"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("true")
)

# Player-performance analysis uses only:
# 1. Games with consistent player point totals.
# 2. Records not marked as DNP.
players_performance = players_cleaned.loc[
    players_cleaned["PLAYER_STATS_VALID"]
    & ~players_cleaned["DNP_FLAG"]
].copy()

# Preserve a separate game-level dataset.
game_columns = [
    "SEASON", "GAME_ID", "GAME_DATE",
    "HOME_TEAM_ID", "HOME_TEAM",
    "AWAY_TEAM_ID", "AWAY_TEAM",
    "HOME_SCORE", "AWAY_SCORE", "HOME_WIN"
]

games_cleaned = (
    data_final[game_columns]
    .drop_duplicates("GAME_ID")
    .merge(
        game_quality[
            ["GAME_ID", "QUALITY_STATUS", "PLAYER_STATS_VALID"]
        ],
        on="GAME_ID",
        how="left",
        validate="one_to_one"
    )
)

# Confirm that all historical game outcomes are retained.
assert len(games_cleaned) == 11979

# Save all cleaned datasets.
games_cleaned.to_csv(
    OUTPUT_DIR / "nba_games_cleaned.csv",
    index=False
)

players_cleaned.to_csv(
    OUTPUT_DIR / "nba_players_cleaned.csv",
    index=False
)

players_performance.to_csv(
    OUTPUT_DIR / "nba_players_performance.csv",
    index=False
)

game_quality.to_csv(
    OUTPUT_DIR / "nba_game_quality_report.csv",
    index=False
)

# Save the source-correction audit for reproducibility.
correction_audit.to_csv(
    OUTPUT_DIR / "nba_correction_audit.csv",
    index=False
)

print("\nFINAL DATASET SUMMARY")
print("=" * 55)
print("Historical games:", len(games_cleaned))
print("Complete box-score games:",
      game_quality["PLAYER_STATS_VALID"].sum())
print("Incomplete box-score games:",
      (~game_quality["PLAYER_STATS_VALID"]).sum())
print("Cleaned player records:", len(players_cleaned))
print("Usable player performance records:",
      len(players_performance))

# Check whether any usable player records still have
# unavailable minutes before calculating per-minute features.
missing_minutes = pd.to_numeric(
    players_performance["MIN"],
    errors="coerce"
).isna()

print("Performance records with unavailable minutes:",
      missing_minutes.sum())

print("\nFiles saved to:", OUTPUT_DIR)


GAME QUALITY SUMMARY


,count
QUALITY_STATUS,
COMPLETE,11492
PLACEHOLDER_BOXSCORE,487



FINAL DATASET SUMMARY
Historical games: 11979
Complete box-score games: 11492
Incomplete box-score games: 487
Cleaned player records: 308226
Usable player performance records: 244588
Performance records with unavailable minutes: 245

Files saved to: /content/drive/MyDrive/Colab Notebooks/Dataset/NBA Dataset/cleaned


## 2.13 Results and interpretation

| Quality-control result | Reported outcome |
|---|---:|
| Historical regular-season games preserved | **11,979** |
| Initial player rows | **308,263** |
| Suspected duplicate ID rows removed | **36** |
| Additional reviewed player row removed | **1** |
| Final retained player rows | **308,226** |
| Team-games with five reported starters after correction | **23,958 / 23,958** |
| Score-validated games (`COMPLETE`) | **11,492** |
| Placeholder box-score games | **487** |
| Records in player-performance subset | **244,588** |
| Player-performance records with unavailable minutes | **245** |

**Interpretation:** All official game outcomes are preserved for team-level EDA. The 487 games with placeholder player box scores should not be treated as zero-contribution performances in rolling player statistics. `players_performance` excludes DNP players and placeholder games, but the remaining 245 records with unavailable minutes still need special handling for minute-normalized analyses.

### Limitations and reproducibility

- Matching team points and counting five starters provide **consistency evidence**, not full verification of every NBA box-score variable or starting lineup.
- The 487 placeholder games cluster in early seasons, potentially affecting player rolling averages and season-to-season comparisons.
- The initial data-preparation code contains two pre-existing references to `combined` rather than `data`. They were deliberately **not edited** here, and should be reviewed before a fresh end-to-end rerun.
- Corrective sources and reasoning should remain traceable in the notebook and the exported audit file.
- Retain the original merged CSV and the complete retained player table for provenance; do not overwrite original raw data.

### Phase 2 conclusion

The phase produced a quality-controlled game-results dataset, a retained player-data reference, a filtered player-performance dataset, and explicit game quality/audit reports. They support **Phase 3: Exploratory Data Analysis**, followed by leakage-free pregame feature engineering and chronological model evaluation.